# Notebook 01 — AOI Delineation & Geospatial Feature Enrichment

**Merged from:** `01_AOI_delineation.ipynb` + `01a_AOI_OSM_JRC_features.ipynb`
**Revised methodology:** Dual-zone AOI (Urban Core + BRA) per MODIFICATION_SPEC §2.1 — GHS-SMOD V2.0
**Implementation version:** `NB01_v5_SMOD`
**Output directory:** `/Results/01_aoi_delineation/`
**CRS:** All spatial operations in `ESRI:102022`; saved in both `ESRI:102022` and `EPSG:4326`

> **Water protocol:** JRC water (occurrence >= 50%) is **subtracted from** all three saved zone
> geometries: UC = UC_raw - water, City-Region = BRA bbox - water, and
> BRA = BRA bbox - UC_raw - water - SMOD 21. The water polygons are saved separately so they can
> be drawn on maps; they are never a zone in their own right. All downstream LST extraction is
> therefore land-only.

> **Zone extents are fixed, not searched:** BRA bbox = `envelope(UC_raw.buffer(10 km))`. The
> equal-area ring search of Yang et al. (2024) was evaluated and **not adopted** — it gave
> degenerate extents for small cities — and was removed from the codebase on 2026-09-28. Each
> city's `zone_metadata_v5.json` records `"bra_method": "fixed_buffer"` accordingly.

---

<a id="nb01-toc"></a>
## Table of Contents

| Section | Description |
|---------|-------------|
| [§1 — Setup & Imports](#nb01-s01) | Python packages, CRS constants, output paths |
| [§1b — GEE Initialisation](#nb01-s01b) | Earth Engine authentication and availability flag |
| [§1c — OSMnx Initialisation](#nb01-s01c) | OSM feature download library |
| [§2 — Load Required Datasets](#nb01-s02) | Cities CSV (UN WUP 2025), Nigeria boundary GeoJSON |
| [§3 — City Selection](#nb01-s03) | Mode-driven city list (`ALL` / `ABOVE_1M` / `TEST` / ...) |
| [§4 — Per-City Dual-Zone AOI Construction](#nb01-s04) | UC / BRA / CR via GHS-SMOD V2.0; smart cache; deduplication |
| [§5 — Export Summary Files](#nb01-s05) | Zone metadata JSON, master GeoJSONs, area summary CSV |
| [§6 — OSM Feature Extraction](#nb01-s06) | Roads + place labels via osmnx, per city |
| [§7 — Three-Zone Study Area Maps](#nb01-s07) | Per-city maps with Cluster25 landscape background |
| [§8 — Nigeria Overview Map](#nb01-s08) | National Cluster25 polygon map + proportional population circles |
| [§9 — Population Trend Charts](#nb01-s09) | Composite + per-city UN WUP population trends (all target cities) |
| [§10 — GEE Export Trigger](#nb01-s10) | Per-city FeatureCollection assets uploaded to GEE |
| [§12 — Summary & Next Steps](#nb01-s12) | Full outputs tree; downstream notebook dependencies |
| [§12 — Predominant Cluster25 per Study City](#nb01-s12) | Area-weighted landscape attribution; cluster summary table; dot map; lookup CSV for NB03 |

---

## Research Objectives

1. Delineate **Urban Core (UC)** — connected component of GHS-SMOD classes 30+23+22 (Urban Centre, Dense Urban Cluster, Semi-dense Urban Cluster) containing the city centroid, with nearby UC components merged within 1 km; JRC water bodies subtracted from final geometry
2. Compute the **Background Reference Area (BRA)** using a **fixed 10 km buffer** around UC_raw (NB01_v5_SMOD); saved geometry = BRA bbox minus UC, water, and SMOD peri-urban (class 21)
3. Construct **City-Region (CR)** = BRA bounding box minus water — full land-only raster extraction domain
4. Export per-city zone files in both CRS (`_v5` suffix) plus zone metadata JSON
5. Extract JRC surface water and OSM features for each city
6. Produce three-zone study area maps with Cluster25 landscape background and Nigeria overview
7. Determine the **predominant Cluster25 physiographic landscape class** per city (§12) via area-weighted spatial intersection of City-Region AOIs against the national landscape character polygons; map city-level cluster attribution

---

## Dual-Zone AOI Construction Workflow

```
City centroid (Lat/Lon)
       │
       ▼
GEE call #1 — GHS-SMOD raster (~165×165 km, 1 km)
       │
       ▼
Connected component of classes 30+23+22 containing centroid
  └─ 1 km fallback search if centroid misses UC pixels
  └─ Merge any other 30/23/22 component within 1 km (SMOD_MERGE_RADIUS_M)
       │
       ▼
UC_raw (before water mask)  ── may be MultiPolygon after merge
       │
       ▼
Fixed 10 km buffer → BRA bbox = UC_raw.buffer(10 km).envelope
       │
GEE call #2 — JRC water within BRA bbox
       │
       ├── UC  = UC_raw  − water
       ├── CR  = BRA bbox − water          (land-only extraction domain)
       │
Reuse SMOD array — vectorise class 21 (peri-urban) within BRA bbox
       │
       ▼
BRA = BRA bbox − UC_raw − water − SMOD peri-urban (class 21)
       │
Save: UC / BRA / CR × 2 CRS + zone_metadata_v5.json
Water bodies overlaid on maps only — not part of any zone.
```

---

## Data Sources

| Dataset | Source | Role |
|---------|--------|------|
| Cities CSV | UN WUP 2025 | City coordinates + population |
| `JRC/GHSL/P2023A/GHS_SMOD_V2-0` | GEE — GHS-SMOD V2.0 ImageCollection | Urban Core delineation |
| `JRC/GSW1_4/GlobalSurfaceWater` | GEE — JRC GSW | Water body mask (subtracted from all zones) |
| `boundary_Nigeria.geojson` | Local | Study area boundary for maps |
| OSM (via osmnx) | OpenStreetMap | Roads + place labels |

**Reference (background; this method was *not* adopted):** Yang et al. (2024). A global urban heat island intensity dataset. *Remote Sensing of Environment*. https://doi.org/10.1016/j.rse.2024.114343


<a id="nb01-s01"></a>
## Section 1 — Setup & Imports

---

> ↑ [Back to Table of Contents](#nb01-toc)

In [ ]:
import sys
import json
import warnings
import time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import rasterio.features
import rasterio.transform
import scipy.ndimage as ndimage
import affine
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import Point, box as shapely_box, mapping
from shapely.ops import unary_union

# Project path setup
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'src'))

import config
from dual_zone_aoi_utils import (
    build_urban_core_smod,
    build_city_region,
    compute_geometry_hash,
    zone_cache_valid,
    save_zone_geojsons,
    build_zone_metadata,
)
from water_download_utils import download_jrc_water_robust

warnings.filterwarnings('ignore')

# ── CRS constants ──────────────────────────────────────────────────────────────
ANALYSIS_CRS = config.ANALYSIS_CRS   # ESRI:102022 — all spatial ops
STORAGE_CRS  = config.STORAGE_CRS    # EPSG:4326  — GeoJSON storage

# ── Output root ────────────────────────────────────────────────────────────────
RESULTS_AOI = project_root / 'Results' / '01_aoi_delineation'
RESULTS_AOI.mkdir(parents=True, exist_ok=True)

print(f"Project root : {project_root}")
print(f"Analysis CRS : {ANALYSIS_CRS}")
print(f"Storage CRS  : {STORAGE_CRS}")
print(f"Results dir  : {RESULTS_AOI}")
print(f"Study Year   : {config.PROJECT_STUDY_YEAR}  (SMOD epoch selection)")


<a id="nb01-s01b"></a>
### 1b — GEE Initialisation

---

> ↑ [Back to Table of Contents](#nb01-toc)

In [ ]:
import ee
from gee_utils import initialize_gee

GEE_AVAILABLE = initialize_gee(
    config.GEE_CONFIG['service_json'],
    config.GEE_CONFIG['project_id'],
)
print(f"GEE available: {GEE_AVAILABLE}")

<a id="nb01-s01c"></a>
### 1c — OSMnx Initialisation

---

> ↑ [Back to Table of Contents](#nb01-toc)

In [ ]:
OSMNX_AVAILABLE = False
try:
    import osmnx as ox
    ox.settings.timeout = 180
    ox.settings.log_console = False
    OSMNX_AVAILABLE = True
    print(f"osmnx {ox.__version__} ready")
except ImportError:
    print("osmnx not installed — OSM features will be skipped")
    print("  Install with: pip install osmnx")

---
<a id="nb01-s02"></a>
## Section 2 — Load Required Datasets

---

> ↑ [Back to Table of Contents](#nb01-toc)

In [ ]:
from aoi_utils import load_aoi_datasets, sort_cities_by_population

# ── Cities CSV + boundary ─────────────────────────────────────────────────────
# L50 urban typology is NOT loaded — UC delineation uses GHS-SMOD (NB01_v4_SMOD).
cities_df, _, boundary_gdf = load_aoi_datasets(
    str(project_root / config.AOI_DATA_PATHS['cities_csv']),
    str(project_root / config.AOI_DATA_PATHS['urban_geojson']),   # ignored (_), kept for API compat
    str(project_root / config.AOI_DATA_PATHS['boundary_geojson']),
)
boundary_gdf = boundary_gdf.to_crs(ANALYSIS_CRS)

print(f"Cities loaded : {len(cities_df)}")
print(f"Boundary CRS  : {boundary_gdf.crs}")


---
<a id="nb01-s03"></a>
## Section 3 — City Selection

---

> ↑ [Back to Table of Contents](#nb01-toc)

In [ ]:
# City selection driven by config.CITY_SELECTION_MODE
# Options: ALL / ABOVE_1M / ABOVE_500K / ABOVE_300K / TEST / SPECIFIC
cities_raw = config.get_cities_to_process()
cities_to_process = sort_cities_by_population(cities_raw, cities_df, year='2024')

print(f"Selection mode : {config.CITY_SELECTION_MODE}")
print(f"Cities to process : {len(cities_to_process)} (sorted by 2024 population)")
print()
for i, city in enumerate(cities_to_process, 1):
    pop = cities_df.loc[cities_df['Urban Agglomeration'] == city, '2024']
    pop_str = f"{float(pop.values[0]):,.0f}k" if len(pop) else "N/A"
    print(f"  {i:3d}. {city:<22s}  {pop_str}")

---
<a id="nb01-s03-filter"></a>
### §3a — Target City Selection Audit

**Purpose:** Document and verify the target city list for this analysis run. The list is determined entirely by `config.CITY_SELECTION_MODE` — changing that one setting re-scopes the entire study across all notebooks automatically.

**Available modes (`config.CITY_SELECTION_MODE`):**

| Mode | Cities included |
|------|-----------------|
| `ALL` | All 791 cities in the WUP Nigeria dataset |
| `ABOVE_1M` | 17 cities with 2024 WUP population ≥ 1,000k |
| `ABOVE_500K` | 37 cities with 2024 WUP population ≥ 500k |
| `ABOVE_300K` | 67 cities with 2024 WUP population ≥ 300k *(current default)* |
| `TEST` | 5 cities for quick development runs |
| `SPECIFIC` | A custom list defined in `config.SELECTED_CITIES` |

**Cross-notebook rule:** This section produces `study_cities_final.csv` (at §11c, after AOI construction and cluster assignment). Every downstream notebook **loads that file** to get the definitive target list — they do not re-implement selection logic. Stakeholder data in NB02 is filtered against the same list; participants from off-target cities are noted in NB02 printouts but excluded from analysis.

> **Note:** Absorbed cities (whose AOI falls entirely within another city's City-Region) are excluded from all spatial analyses. They are documented in §5 and carried in `study_cities_final.csv` with `status = absorbed`.

In [ ]:
# §3a — Target city selection audit (config-driven)
# Prints the full target list for the current CITY_SELECTION_MODE.
# No threshold is hardcoded here — the list comes entirely from config.

_mode = config.CITY_SELECTION_MODE

# Population threshold implied by mode (for consistency checks only)
_threshold_k = {'ABOVE_1M': 1_000, 'ABOVE_500K': 500, 'ABOVE_300K': 300}.get(_mode)

print("=" * 60)
print(f"  City selection mode  : {_mode}")
if _threshold_k:
    print(f"  Population threshold : ≥ {_threshold_k:,}k (2024 WUP)")
elif _mode == 'SPECIFIC':
    print(f"  Custom list size     : {len(config.SELECTED_CITIES)} cities")
elif _mode == 'TEST':
    print(f"  (Test set — not for production analysis)")
print(f"  Target cities        : {len(cities_to_process)}")
print("=" * 60)
print()

print(f"{'#':>3}  {'City':<24}  {'Pop 2024 (k)':>13}")
print("─" * 45)
for _i, _city in enumerate(cities_to_process, 1):
    _pop_row = cities_df.loc[cities_df['Urban Agglomeration'] == _city, '2024']
    _pop_k   = float(_pop_row.values[0]) if len(_pop_row) else 0
    print(f"{_i:>3}. {_city:<24}  {_pop_k:>11,.0f}k")

# Internal consistency check: if a threshold mode, flag any city below it
if _threshold_k:
    _violations = []
    for _city in cities_to_process:
        _r = cities_df.loc[cities_df['Urban Agglomeration'] == _city, '2024']
        if len(_r) and float(_r.values[0]) < _threshold_k:
            _violations.append((_city, float(_r.values[0])))
    print()
    if _violations:
        print(f"⚠  CONFIG INCONSISTENCY — {len(_violations)} cities are below the {_threshold_k}k threshold:")
        for _c, _pk in _violations:
            print(f"   {_c}  ({_pk:.0f}k)  — check config.py")
    else:
        print(f"✓  All {len(cities_to_process)} target cities are consistent with mode '{_mode}'.")

print()
print("► Absorbed cities will be documented in §5.")

---
<a id="nb01-s04"></a>
## Section 4 - Per-City Dual-Zone AOI Construction (GHS-SMOD)

**Purpose:** Construct three nested, water-free study zones for each city — Urban Core (UC), Background Reference Area (BRA), and City-Region (CR) — from the GHS Settlement Model raster. These zones are the spatial foundation for all LST, population, and hotspot analyses in NB03–NB06.

**Why UC = GHS-SMOD classes 30 + 23 + 22?**
GHS-SMOD class 30 (*Urban Centre*), 23 (*Dense Urban Cluster*), and 22 (*Semi-dense Urban Cluster*) together represent the built-up, high-density settlement fabric where urban heat accumulation is most pronounced. Classes below 22 (suburban fringes, low-density rural clusters) are excluded because they do not exhibit consistent SUHI signatures in West African cities and would inflate the UC footprint with ambiguous land cover.

**Why a 10 km fixed BRA buffer?**
A 10 km radial buffer around the UC captures the near-field rural surroundings at a scale that: (a) encompasses the dominant land-surface types controlling rural LST (vegetation, bare soil, water) without reaching the thermal footprint of adjacent cities; and (b) is consistent with published SUHI literature for sub-Saharan Africa, making SUHII values comparable across studies. Coastal and water pixels are removed from the BRA so that the thermal contrast is land-surface only.

**Inputs:**
- GEE: `JRC/GHSL/P2023A/GHS_SMOD_E2020_GLOBE_R2023A_54009_1000_V1_0` (1 km, SMOD 2020)
- GEE: `JRC/GSW1_4/GlobalSurfaceWater` (JRC water mask, occurrence ≥ 50%)
- `cities_to_process` — ordered city list from §3

**Method:** For each city (population-ordered, largest first):

| Step | Action |
|------|--------|
| **4a** | Download GHS-SMOD raster within ~165 km envelope (GEE call #1) |
| **4b** | Extract UC_raw — connected component of classes 30+23+22; 1 km fallback if centroid misses; merge nearby 30/23/22 components within 1 km |
| **4c** | Fixed 10 km buffer — BRA bbox = `UC_raw.buffer(10 km).envelope` (rectangle) |
| **4d** | Download JRC water within BRA bbox (GEE call #2, tiled robust downloader) |
| **4e** | Apply water mask — UC = UC_raw − water; CR = BRA bbox − water |
| **4f** | Extract SMOD peri-urban (class 21) within BRA bbox (reuse SMOD array from 4a) |
| **4g** | Finalise BRA = BRA bbox − UC_raw − water − SMOD peri-urban |
| **4h** | Compute zone fractions (water, SMOD built-up, effective area) for UC / BRA / CR |
| **4i** | Deduplication — absorb any city whose centroid falls inside current **City-Region** |
| **4j** | Save metadata + all 6 GeoJSONs (UC / BRA / CR × 2 CRS, `_v5` suffix) |

**All three saved zones are water-free.** Water bodies are saved separately and overlaid on maps only.
**Smart cache:** Skips a city if `_v5` zone files exist and `source_version + geometry_hash` match (`NB01_v5_SMOD`).
**Deduplication:** Smaller cities whose centroid falls inside a previously processed City-Region are absorbed and skipped.

**Outputs:**
- Per-city zone GeoJSONs under `Results/01_aoi_delineation/<CityName>/`
- `zone_area_summary.csv` — UC / BRA / CR areas, fractions, and absorption status per city

---

> ↑ [Back to Table of Contents](#nb01-toc)

In [ ]:
from shapely.geometry import shape as shapely_shape

# ── State tracking ─────────────────────────────────────────────────────────────
all_zone_results = {}
absorbed_cities  = {}
cities_set       = set(cities_to_process)
SOURCE_VERSION   = 'NB01_v5_SMOD'
FILE_VERSION     = '_v5'

# ── SMOD epoch: select year closest to config.PROJECT_STUDY_YEAR ───────────────
# GHS-SMOD V2.0 is an ImageCollection at JRC/GHSL/P2023A/GHS_SMOD_V2-0.
# Filter by calendarRange to get the image for the target epoch year.
SMOD_EPOCH = min(config.SMOD_AVAILABLE_EPOCHS,
                 key=lambda y: abs(y - config.PROJECT_STUDY_YEAR))
if GEE_AVAILABLE:
    _smod_col = ee.ImageCollection(config.SMOD_GEE_ASSET)
    smod_img  = _smod_col.filter(
        ee.Filter.calendarRange(SMOD_EPOCH, SMOD_EPOCH, 'year')
    ).first().select('smod_code')
    print(f"SMOD epoch  : {SMOD_EPOCH}  (PROJECT_STUDY_YEAR={config.PROJECT_STUDY_YEAR})")
    print(f"Collection  : {config.SMOD_GEE_ASSET}")
else:
    smod_img = None
    print(f"GEE unavailable — SMOD download will be skipped (epoch would be {SMOD_EPOCH})")

processed = 0

for idx, current_city in enumerate(cities_to_process, 1):

    # ── Deduplication check ───────────────────────────────────────────────────
    if current_city in absorbed_cities:
        print(f"Skipped [{idx}/{len(cities_to_process)}] {current_city} "
              f"-> absorbed by '{absorbed_cities[current_city]}'")
        continue

    processed += 1
    city_slug = current_city.replace(' ', '_')
    city_dir  = RESULTS_AOI / city_slug
    city_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*64}")
    print(f"[{idx}/{len(cities_to_process)}] {current_city}")
    print(f"{'='*64}")

    city_row   = cities_df[cities_df['Urban Agglomeration'] == current_city].iloc[0]
    city_point = Point(city_row['Longitude'], city_row['Latitude'])
    city_pt_gdf = gpd.GeoDataFrame(
        [{'city': current_city}], geometry=[city_point], crs=STORAGE_CRS
    ).to_crs(ANALYSIS_CRS)
    city_pt_102022 = city_pt_gdf.geometry.iloc[0]

    # ── Smart cache ────────────────────────────────────────────────────────────
    uc_4326_path = city_dir / f"{city_slug}_urban_core_4326_v5.geojson"
    cache_meta   = city_dir / f"{city_slug}_zone_metadata_v5.json"
    if zone_cache_valid(uc_4326_path, cache_meta, SOURCE_VERSION, city_pt_102022):
        print("  [CACHE] Loading saved zones")
        meta    = json.loads(cache_meta.read_text())
        uc_gdf  = gpd.read_file(city_dir / f"{city_slug}_urban_core_4326_v5.geojson")
        bra_gdf = gpd.read_file(city_dir / f"{city_slug}_bra_4326_v5.geojson")
        cr_gdf  = gpd.read_file(city_dir / f"{city_slug}_city_region_4326_v5.geojson")
        all_zone_results[current_city] = {
            'uc_gdf': uc_gdf, 'bra_gdf': bra_gdf, 'cr_gdf': cr_gdf,
            'metadata': meta, 'city_slug': city_slug, 'city_dir': city_dir,
            'city_row': city_row,
        }
        for c in meta.get('absorbed_cities', []):
            if c != current_city and c in cities_set:
                absorbed_cities[c] = current_city
        continue

    # ── Step 4a: Download GHS-SMOD raster (GEE call #1) ──────────────────────
    print("  Step 4a -- GHS-SMOD raster download")
    smod_array     = None
    smod_transform = None
    smod_crs_str   = STORAGE_CRS

    if GEE_AVAILABLE and smod_img is not None:
        try:
            cx    = float(city_row['Longitude'])
            cy    = float(city_row['Latitude'])
            r_deg = config.SMOD_DOWNLOAD_RADIUS_M / 111_000
            west, south, east, north = cx - r_deg, cy - r_deg, cx + r_deg, cy + r_deg

            region           = ee.Geometry.BBox(west, south, east, north)
            smod_reprojected = smod_img.reproject(crs=smod_crs_str, scale=1000)
            result           = smod_reprojected.sampleRectangle(region=region, defaultValue=0)
            smod_array       = np.array(result.get('smod_code').getInfo(), dtype=np.int16)

            n_rows, n_cols   = smod_array.shape
            px_w = (east  - west)  / n_cols
            px_h = (north - south) / n_rows
            smod_transform   = affine.Affine(px_w, 0, west, 0, -px_h, north)

            print(f"    SMOD array : {smod_array.shape}  "
                  f"unique classes: {np.unique(smod_array).tolist()}")
        except Exception as e:
            print(f"    SMOD download failed: {e}")

    # ── Step 4b: Urban Core (raw, before water mask) ──────────────────────────
    print("  Step 4b -- UC_raw (GHS-SMOD classes 30+23+22)")
    if smod_array is not None:
        uc_geom_raw, uc_flags = build_urban_core_smod(
            city_point=city_pt_102022,
            smod_array=smod_array,
            smod_transform=smod_transform,
            smod_crs=smod_crs_str,
            analysis_crs=ANALYSIS_CRS,
            uc_classes=config.SMOD_UC_CLASSES,
            fallback_radius_m=config.SMOD_FALLBACK_RADIUS_M,
            merge_radius_m=config.SMOD_MERGE_RADIUS_M,
        )
    else:
        uc_geom_raw = city_pt_102022.buffer(5_000)
        uc_flags    = ['gee_unavailable', 'fallback_buffer']
        print("    GEE unavailable — using 5 km fallback circle")

    uc_raw_area_km2 = uc_geom_raw.area / 1e6
    print(f"    UC_raw area : {uc_raw_area_km2:.2f} km²  flags: {uc_flags}")

    # ── Step 4c: BRA bbox — fixed buffer around UC_raw ────────────────────────
    print(f"  Step 4c -- BRA bbox (fixed {config.BRA_BUFFER_M/1000:.0f} km buffer, NB01_v5_SMOD)")
    bra_bbox_geom = uc_geom_raw.buffer(config.BRA_BUFFER_M).envelope
    print(f"    BRA bbox : {bra_bbox_geom.area/1e6:.2f} km²  buffer : {config.BRA_BUFFER_M/1000:.0f} km")

    # ── Step 4d: JRC water within BRA bbox (GEE call #2) ─────────────────────
    print("  Step 4d -- JRC water download (BRA bbox)")
    water_geom_102022 = None
    water_file = city_dir / f"{city_slug}_water_bodies_4326.geojson"

    if water_file.exists():
        w_cached = gpd.read_file(water_file).to_crs(ANALYSIS_CRS)
        if not w_cached.empty:
            water_geom_102022 = w_cached.geometry.unary_union
            # Backfill 102022 file if missing (upgrade existing cities)
            water_file_102022 = city_dir / f"{city_slug}_water_bodies_102022.geojson"
            if not water_file_102022.exists():
                w_cached.to_file(water_file_102022, driver='GeoJSON')
            print(f"    Water : {len(w_cached)} features (cached)")
        else:
            print("    Water : cache empty")
    elif GEE_AVAILABLE:
        w_gdf = download_jrc_water_robust(
            search_geom=bra_bbox_geom,
            search_crs=ANALYSIS_CRS,
            occurrence_threshold=config.JRC_OCCURRENCE_THRESHOLD,
            tile_size_m=config.JRC_TILE_SIZE_M,
            min_area_m2=config.JRC_MIN_WATER_AREA_M2,
            verbose=True,
        )
        if w_gdf is not None and not w_gdf.empty:
            w_gdf.to_file(water_file, driver='GeoJSON')
            # Also save in ANALYSIS_CRS (102022) for raster operations (SEVIRI masking, etc.)
            water_file_102022 = city_dir / f"{city_slug}_water_bodies_102022.geojson"
            w_gdf.to_crs(ANALYSIS_CRS).to_file(water_file_102022, driver='GeoJSON')
            water_geom_102022 = w_gdf.to_crs(ANALYSIS_CRS).geometry.unary_union
            print(f"    Water : {len(w_gdf)} features saved")
        else:
            print("    Water : none found or download failed")
    else:
        print("    Water : GEE unavailable")

    # ── Step 4e: Apply water mask to UC and compute CR ────────────────────────
    # UC and CR must not contain any water bodies (LST is land-only).
    # Water bodies are saved separately and overlaid on maps only.
    print("  Step 4e -- Apply water mask → UC and City-Region (land-only zones)")
    if water_geom_102022 is not None and not water_geom_102022.is_empty:
        uc_geom = uc_geom_raw.difference(water_geom_102022).buffer(0)
        cr_geom = build_city_region(bra_bbox_geom, water_geom_102022)
    else:
        uc_geom = uc_geom_raw
        cr_geom = build_city_region(bra_bbox_geom)

    uc_area_km2 = uc_geom.area / 1e6
    cr_area_km2 = cr_geom.area / 1e6
    print(f"    UC  : {uc_raw_area_km2:.2f} km² (raw) → {uc_area_km2:.2f} km² (water-cleaned)")
    print(f"    CR  : {cr_area_km2:.2f} km² (water-cleaned)")

    # ── Step 4f: SMOD class 21 mask within BRA bbox (reuse array from 4a) ────
    print("  Step 4f -- SMOD built-up mask (class 21 within BRA bbox)")
    smod_builtup_geom = None

    if smod_array is not None:
        try:
            builtup_mask = np.isin(smod_array, config.SMOD_BUILTUP_CLASSES).astype(np.uint8)
            if builtup_mask.sum() > 0:
                raw_shapes   = list(rasterio.features.shapes(builtup_mask, transform=smod_transform))
                builtup_polys = [shapely_shape(g) for g, v in raw_shapes if v == 1]
                if builtup_polys:
                    bu_gdf = gpd.GeoDataFrame(
                        geometry=builtup_polys, crs=smod_crs_str
                    ).to_crs(ANALYSIS_CRS)
                    bra_bbox_gdf = gpd.GeoDataFrame(geometry=[bra_bbox_geom], crs=ANALYSIS_CRS)
                    bu_clipped   = gpd.clip(bu_gdf, bra_bbox_gdf)
                    if not bu_clipped.empty:
                        smod_builtup_geom = bu_clipped.geometry.unary_union
                        print(f"    SMOD built-up in BRA bbox : {smod_builtup_geom.area/1e6:.2f} km²")
                    else:
                        print("    SMOD built-up : none within BRA bbox")
            else:
                print("    SMOD built-up : no built-up pixels in raster")
        except Exception as e:
            print(f"    SMOD built-up extraction failed: {e}")
    else:
        print("    SMOD built-up : no raster available")

    # ── Step 4g: Finalise BRA (bbox minus UC_raw, water, SMOD class 21) ──────
    # UC_raw used here (not UC_clean) to exclude the full urban footprint area.
    print("  Step 4g -- Finalise BRA (BRA bbox − UC − water − SMOD built-up)")
    bra_geom = bra_bbox_geom.difference(uc_geom_raw)
    if water_geom_102022 is not None and not water_geom_102022.is_empty:
        bra_geom = bra_geom.difference(water_geom_102022)
    if smod_builtup_geom is not None and not smod_builtup_geom.is_empty:
        bra_geom = bra_geom.difference(smod_builtup_geom)
    bra_geom = bra_geom.buffer(0)

    bra_area_km2 = bra_geom.area / 1e6
    bra_uc_ratio = bra_area_km2 / uc_area_km2 if uc_area_km2 > 0 else float('nan')
    print(f"    BRA effective area : {bra_area_km2:.2f} km²  BRA/UC ratio : {bra_uc_ratio:.3f}")

    # ── Step 4h: Zone fractions ───────────────────────────────────────────────
    def _zone_fracs(zone_geom, water_union, builtup_union, zone_name):
        raw = zone_geom.area
        if raw == 0:
            return {}
        w_area = (zone_geom.intersection(water_union).area
                  if water_union is not None and not water_union.is_empty else 0.0)
        b_area = (zone_geom.intersection(builtup_union).area
                  if builtup_union is not None and not builtup_union.is_empty else 0.0)
        eff = max(0.0, raw - w_area - b_area)
        return {
            f"{zone_name}_water_area_km2"       : round(w_area / 1e6, 4),
            f"{zone_name}_water_frac"            : round(w_area / raw, 4),
            f"{zone_name}_smod_builtup_area_km2" : round(b_area / 1e6, 4),
            f"{zone_name}_smod_builtup_frac"     : round(b_area / raw, 4),
            f"{zone_name}_effective_area_km2"    : round(eff   / 1e6, 4),
        }

    zone_fractions = {}
    zone_fractions.update(_zone_fracs(uc_geom_raw,   water_geom_102022, smod_builtup_geom, 'uc'))
    zone_fractions.update(_zone_fracs(bra_bbox_geom, water_geom_102022, smod_builtup_geom, 'bra'))
    zone_fractions.update(_zone_fracs(cr_geom,       water_geom_102022, smod_builtup_geom, 'cr'))
    print(f"    CR effective area : {zone_fractions.get('cr_effective_area_km2','n/a')} km²")

    # ── Step 4i: Deduplication ────────────────────────────────────────────────
    # A city is absorbed if its centroid falls inside the current City-Region.
    # CR is the full study extent — overlapping CRs would create duplicate LST analysis.
    absorbed_now = []
    for other_city in cities_to_process:
        if other_city == current_city or other_city in absorbed_cities:
            continue
        other_row = cities_df[cities_df['Urban Agglomeration'] == other_city]
        if other_row.empty:
            continue
        other_pt  = Point(float(other_row['Longitude'].values[0]),
                          float(other_row['Latitude'].values[0]))
        other_gdf = gpd.GeoDataFrame(geometry=[other_pt], crs=STORAGE_CRS).to_crs(ANALYSIS_CRS)
        if cr_geom.contains(other_gdf.geometry.iloc[0]):
            absorbed_cities[other_city] = current_city
            absorbed_now.append(other_city)
    if absorbed_now:
        print(f"    Absorbed : {absorbed_now}")

    # ── Step 4j: Metadata + Save ──────────────────────────────────────────────
    meta = build_zone_metadata(
        city_name=current_city, city_row=city_row,
        uc_geom=uc_geom, bra_geom=bra_geom, cr_geom=cr_geom,
        bra_buffer_m=config.BRA_BUFFER_M, uc_flags=uc_flags,
        absorbed_cities=absorbed_now,
        source_version=SOURCE_VERSION,
        smod_epoch_year=SMOD_EPOCH,
        analysis_crs=ANALYSIS_CRS,
    )
    meta['uc_raw_area_km2'] = round(uc_raw_area_km2, 4)   # record pre-water-mask area
    meta.update(zone_fractions)

    save_zone_geojsons(
        city_dir=city_dir, city_slug=city_slug,
        uc_geom=uc_geom, bra_geom=bra_geom, cr_geom=cr_geom,
        meta=meta, analysis_crs=ANALYSIS_CRS, storage_crs=STORAGE_CRS,
        file_version=FILE_VERSION,
    )

    uc_gdf  = gpd.read_file(city_dir / f"{city_slug}_urban_core_4326_v5.geojson")
    bra_gdf = gpd.read_file(city_dir / f"{city_slug}_bra_4326_v5.geojson")
    cr_gdf  = gpd.read_file(city_dir / f"{city_slug}_city_region_4326_v5.geojson")

    all_zone_results[current_city] = {
        'uc_gdf': uc_gdf, 'bra_gdf': bra_gdf, 'cr_gdf': cr_gdf,
        'metadata': meta, 'city_slug': city_slug, 'city_dir': city_dir,
        'city_row': city_row,
    }
    print(f"  [DONE] {current_city}")

# ── Summary ────────────────────────────────────────────────────────────────────
print(f"\n{'='*64}")
print(f"Candidates      : {len(cities_to_process)}")
print(f"Unique AOIs     : {len(all_zone_results)}")
print(f"Absorbed cities : {len(absorbed_cities)}")
if absorbed_cities:
    print("  Absorbed -> Parent:")
    for c, p in absorbed_cities.items():
        print(f"    {c} -> {p}")
print(f"SMOD epoch used : {SMOD_EPOCH}")
print(f"Source version  : {SOURCE_VERSION}")
print(f"{'='*64}")


---
<a id="nb01-s05"></a>
## Section 5 — Export Summary Files

Saves:  
- `Results/01_aoi_delineation/absorbed_cities.csv` — deduplication log  
- `Results/01_aoi_delineation/all_cities_zone_metadata.json` — aggregated zone stats  
- `Results/01_aoi_delineation/all_cities_zones_4326.geojson` — master GeoJSON (all three zones, all cities)  
- `Results/01_aoi_delineation/all_cities_zones_102022.geojson` — same in Analysis CRS  
- `Results/01_aoi_delineation/zone_area_summary.csv` — per-city area table

---

> ↑ [Back to Table of Contents](#nb01-toc)

In [ ]:
# ── absorbed_cities.csv ────────────────────────────────────────────────────────
absorbed_df = pd.DataFrame(
    [{'absorbed_city': c, 'parent_aoi': p} for c, p in absorbed_cities.items()]
)
absorbed_path = RESULTS_AOI / 'absorbed_cities.csv'
absorbed_df.to_csv(absorbed_path, index=False)
print(f"Saved absorbed_cities.csv ({len(absorbed_df)} rows)")

# ── all_cities_zone_metadata.json ─────────────────────────────────────────────
all_meta = {city: res['metadata'] for city, res in all_zone_results.items()}
meta_path = RESULTS_AOI / 'all_cities_zone_metadata.json'
meta_path.write_text(json.dumps(all_meta, indent=2, default=str))
print(f"Saved all_cities_zone_metadata.json ({len(all_meta)} cities)")

# ── zone_area_summary.csv ─────────────────────────────────────────────────────
area_rows = []
for city, res in all_zone_results.items():
    m = res['metadata']
    area_rows.append({
        'city':                     city,
        'uc_area_km2':              m.get('uc_area_km2'),
        'bra_area_km2':             m.get('bra_area_km2'),
        'cr_area_km2':              m.get('cr_area_km2'),
        'bra_uc_ratio':             m.get('bra_uc_ratio'),
        'uc_flags':                 ';'.join(m.get('uc_flags', [])),
        'smod_epoch_year':          m.get('smod_epoch_year'),
        'bra_water_frac':           m.get('bra_water_frac'),
        'bra_smod_builtup_frac':    m.get('bra_smod_builtup_frac'),
        'bra_effective_area_km2':   m.get('bra_effective_area_km2'),
        'absorbed_by':              absorbed_cities.get(city, ''),
    })
area_df = pd.DataFrame(area_rows).sort_values('uc_area_km2', ascending=False)
area_df.to_csv(RESULTS_AOI / 'zone_area_summary.csv', index=False)
print(f"Saved zone_area_summary.csv ({len(area_df)} cities)")

# ── Master GeoJSON — all three zones combined ──────────────────────────────────
zone_rows = []
for city, res in all_zone_results.items():
    for zone_name, gdf in [('urban_core', res['uc_gdf']),
                            ('bra',         res['bra_gdf']),
                            ('city_region', res['cr_gdf'])]:
        row = gdf.copy()
        row['city'] = city
        row['zone'] = zone_name
        zone_rows.append(row)

master_gdf_4326 = gpd.GeoDataFrame(pd.concat(zone_rows, ignore_index=True), crs=STORAGE_CRS)
master_gdf_4326.to_file(RESULTS_AOI / 'all_cities_zones_4326.geojson', driver='GeoJSON')

master_gdf_102022 = master_gdf_4326.to_crs(ANALYSIS_CRS)
master_gdf_102022.to_file(RESULTS_AOI / 'all_cities_zones_102022.geojson', driver='GeoJSON')

print(f"Saved master GeoJSONs ({len(all_zone_results)} cities × 3 zones)")


---
<a id="nb01-s06"></a>
## Section 6 — OSM Feature Extraction

Downloads roads (OSM drive network) and place labels for each city using the **City-Region** as the query polygon.  
Results saved to `Results/01_aoi_delineation/{city_slug}/osm/`.

---

> ↑ [Back to Table of Contents](#nb01-toc)

In [ ]:
for city, res in all_zone_results.items():
    city_slug = res['city_slug']
    city_dir  = res['city_dir']
    osm_dir   = city_dir / 'osm'
    osm_dir.mkdir(parents=True, exist_ok=True)

    cr_4326  = res['cr_gdf'].to_crs(STORAGE_CRS)
    
    # Repair geometries to handle any topological issues from masking operations
    cr_geoms_valid = [geom.buffer(0) if not geom.is_valid else geom for geom in cr_4326.geometry]
    
    # Use convex hull for Overpass query — avoids missed features at water edges.
    # Clip downloaded results to the actual CR polygon afterward.
    try:
        query_poly = unary_union(cr_geoms_valid).convex_hull
        cr_poly    = unary_union(cr_geoms_valid)
    except Exception as e:
        print(f"  {city}: CR topology error, using convex hull fallback")
        query_poly = unary_union(cr_geoms_valid).convex_hull
        cr_poly    = query_poly

    print(f"\n{city}")

    # ── Roads ──────────────────────────────────────────────────────────────────
    roads_file = osm_dir / f"{city_slug}_roads_4326.geojson"
    if roads_file.exists():
        print(f"  Roads  : cached ({roads_file.name})")
    elif OSMNX_AVAILABLE:
        try:
            print(f"  Roads  : downloading (major roads only)...")
            # Only request major roads: motorway, trunk, primary, secondary, tertiary
            # This reduces Overpass API response size dramatically
            roads = ox.features_from_polygon(query_poly, tags={
                'highway': ['motorway', 'trunk', 'primary', 'secondary', 'tertiary']
            })
            
            if not roads.empty:
                roads = roads.to_crs(STORAGE_CRS)
                roads = roads[roads.geometry.intersects(cr_poly)]   # clip to CR
                roads.to_file(roads_file, driver='GeoJSON')
                print(f"  Roads  : ✓ {len(roads)} edges saved")
            else:
                print("  Roads  : ⊘ empty result")
        except Exception as e:
            print(f"  Roads  : ⚠️ {type(e).__name__}: {str(e)[:80]}")
    else:
        print("  Roads  : osmnx unavailable")

    # ── Places ─────────────────────────────────────────────────────────────────
    places_file = osm_dir / f"{city_slug}_places_4326.geojson"
    if places_file.exists():
        print(f"  Places : cached ({places_file.name})")
    elif OSMNX_AVAILABLE:
        try:
            print(f"  Places : downloading (town/suburb only)...")
            # Only request specific place types needed for visualization (not ALL places)
            # This reduces Overpass API response size dramatically for large cities
            places = ox.features_from_polygon(query_poly, tags={
                'place': ['town', 'suburb', 'village', 'hamlet']
            })
            
            if not places.empty:
                places = places.to_crs(STORAGE_CRS)
                places = places[places.geometry.intersects(cr_poly)]   # clip to CR
                places.to_file(places_file, driver='GeoJSON')
                print(f"  Places : ✓ {len(places)} features saved")
            else:
                print("  Places : ⊘ empty result")
        except Exception as e:
            print(f"  Places : ⚠️ {type(e).__name__}: {str(e)[:80]}")
    else:
        print("  Places : osmnx unavailable")

print(f"\n✓ OSM feature extraction complete ({len(all_zone_results)} cities)")

---
<a id="nb01-s07"></a>
## Section 7 — Three-Zone Study Area Maps

**Purpose:** Generate a per-city reference map showing all three analytical zones against physiographic and geographic context.

**Inputs:** `all_zone_results` (non-absorbed cities only), Cluster25 landscape GeoJSON, per-city OSM layers (roads, places), per-city water bodies.

**Method:** Three rendering passes keep zone fills, geographic features, and zone outlines at distinct depth levels so no element buries another.

| Pass | Layers | zorder | Notes |
|------|--------|--------|-------|
| **1 — Fills** | CR fill | 1 | Very light (#f0f0eb, α=0.30) — extent only |
| | Cluster25 patches | 2 | Canonical colours, clipped to extent, α=0.50 |
| | BRA fill | 3 | Gold (#f5c842, α=0.28) — BRA extent |
| **2 — Features** | Water bodies | 4 | Steelblue, α=0.80 |
| | UC fill | 5 | Orange (#e07b00, α=0.72) — primary focus zone |
| **3 — Outlines** | BRA outline | 6 | Dashed brown |
| | UC outline | 7 | Solid dark brown |
| | CR outline | 8 | Solid dark grey — frames the full extent |
| | Cluster25 C{N} labels | 9 | Deferred so labels sit above outlines |
| **4 — Top** | Roads | 10.5 | Grey (#888), above all zone elements |
| | Place dots | 11 | Suburbs/towns (darkred markers) |
| | Place labels | 12 | Name annotations with white background |
| | **Legend** | **50** | Pinned inside CR bounds — always on top |

Map extent = convex hull of City-Region (passed as `aoi_gdf`).
Title shows: `{city} — Three-Zone Study Area | UC: X km² | BRA: X km² | CR: X km²`

✅ SPEC CONFIRMED (§2.1): All three zones visible per city map.

---

> ↑ [Back to Table of Contents](#nb01-toc)

In [ ]:
from visualization import create_geospatial_features_map
import geopandas as gpd

# ── Load Cluster25 landscape once (clipping happens inside the function) ───────
_land_path = config.LANDSCAPE_DATA_PATH
landscape_gdf = gpd.read_file(_land_path).to_crs(STORAGE_CRS) if Path(_land_path).exists() else None
if landscape_gdf is not None:
    landscape_gdf['geometry'] = landscape_gdf.geometry.buffer(0)
    print(f"Landscape loaded: {len(landscape_gdf)} polygons, "
          f"{landscape_gdf['Cluster25'].nunique()} clusters")
else:
    print("WARNING: landscape GeoJSON not found — Cluster25 layer will be skipped")

# ── Set force_regenerate = True to overwrite existing maps ────────────────────
# Set to False once all maps are up to date.
force_regenerate = True

def _reproj_repair(gdf, crs):
    out = gdf.to_crs(crs).copy()
    out['geometry'] = out.geometry.buffer(0)
    return out

print("Generating per-city three-zone maps...")
skipped = created = 0

for city, res in all_zone_results.items():
    city_slug = res['city_slug']
    city_dir  = res['city_dir']
    fig_dir   = city_dir / 'figures'
    fig_dir.mkdir(parents=True, exist_ok=True)

    fig_path = fig_dir / f"{city_slug}_three_zone_map.svg"
    if fig_path.exists() and not force_regenerate:
        skipped += 1
        continue

    # ── Load feature layers ────────────────────────────────────────────────────
    def _load_or_none(p):
        p = Path(p)
        if p.exists():
            gdf = gpd.read_file(p)
            return gdf if not gdf.empty else None
        return None

    water_gdf  = _load_or_none(city_dir / f"{city_slug}_water_bodies_4326.geojson")
    osm_dir    = city_dir / 'osm'
    roads_gdf  = _load_or_none(osm_dir / f"{city_slug}_roads_4326.geojson")
    places_gdf = _load_or_none(osm_dir / f"{city_slug}_places_4326.geojson")

    # ── Reproject + repair all three zones ────────────────────────────────────
    uc_4326  = _reproj_repair(res['uc_gdf'],  STORAGE_CRS)
    bra_4326 = _reproj_repair(res['bra_gdf'], STORAGE_CRS)
    cr_4326  = _reproj_repair(res['cr_gdf'],  STORAGE_CRS)

    # Map extent = convex hull of CR (clean rectangular neatline)
    extent_4326 = gpd.GeoDataFrame(
        geometry=[cr_4326.geometry.unary_union.convex_hull], crs=STORAGE_CRS
    )

    try:
        fig = create_geospatial_features_map(
            city_name=city,
            aoi_gdf=extent_4326,              # neatline / map extent
            cr_gdf=cr_4326,                   # City-Region — light background
            main_polygon_gdf=uc_4326,         # Urban Core (UC) — orange
            adjacent_polygons_gdf=bra_4326,   # BRA — gold dashed
            landscape_gdf=landscape_gdf,      # Cluster25 background + labels
            cluster25_colors=config.CLUSTER25_COLORS,
            water_gdf=water_gdf,
            roads_gdf=roads_gdf,
            places_gdf=places_gdf,
            place_types_filter=['town', 'suburb'],
            label_buffer_km=1.4,
            figsize=(12, 10),
            save_path=str(fig_path),
            dpi=config.FIGURE_DPI,
        )
        plt.close(fig)
        created += 1
    except Exception as e:
        print(f"  WARNING {city}: {e}")

print(f"Done — {created} created, {skipped} cached.")


---
<a id="nb01-s08"></a>
## Section 8 — Nigeria Overview Map

**Purpose:** Plot all study cities on the Cluster25 physiographic landscape background, with proportional population circles and city-name labels.

**City set:** Non-absorbed study cities only (from `all_zone_results`). Absorbed cities do not get separate dots; their WUP population is added to their parent city so the circle size reflects the full urban agglomeration.

**Population roll-up:** For each absorbed city its 2024 WUP population is summed into the parent city's total before scaling the circles. Example: if Ikorodu is absorbed by Lagos, Lagos's dot shows Lagos + Ikorodu combined population.

**Inputs:**
- `all_zone_results` — non-absorbed city zone GDFs (from §4)
- `absorbed_cities` — dict `{absorbed_city: parent_city}` (built in §4/§5)
- `cities_df` — 2024 WUP population table (from §2)

**Outputs:**
- `Results/01_aoi_delineation/Nigeria_study_area_overview.svg/.pdf/.png`
- `Results/01_aoi_delineation/Nigeria_study_area_legend.svg/.pdf/.png`

---

> ↑ [Back to Table of Contents](#nb01-toc)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines  as mlines
import numpy as np
import pandas as pd
from pathlib import Path
from visualization import create_nigeria_cities_landscape_map

# ── 8a: Build population dict with absorbed cities rolled up to parent ────────
_yr = str(config.PROJECT_STUDY_YEAR)

# Base WUP population for all target cities (thousands → actual)
_all_pop = {}
for _, _r in cities_df.iterrows():
    _city = _r['Urban Agglomeration']
    if _city in cities_to_process and pd.notna(_r.get(_yr)):
        _all_pop[_city] = float(_r[_yr]) * 1_000

# Start with non-absorbed cities only
_pop_rollup = {c: _all_pop.get(c, 0) for c in all_zone_results}

# Roll absorbed city populations up to parent
_n_absorbed = len(absorbed_cities)
for _absorbed, _parent in absorbed_cities.items():
    if _parent in _pop_rollup and _absorbed in _all_pop:
        _pop_rollup[_parent] = _pop_rollup[_parent] + _all_pop[_absorbed]

if _n_absorbed:
    print(f"Population roll-up: {_n_absorbed} absorbed cities added to parent totals")
    for _absorbed, _parent in absorbed_cities.items():
        _add = _all_pop.get(_absorbed, 0)
        print(f"  {_absorbed:22s} -> {_parent}  (+{_add/1_000:,.0f}k => total {_pop_rollup.get(_parent,0)/1_000:,.0f}k)")

_n_shown = len(all_zone_results)
print()
print(f"Map will show {_n_shown} cities ({_n_absorbed} absorbed, population rolled up)")

# ── 8b: AOI dict (City-Region boundaries, non-absorbed cities only) ───────────
aoi_dict = {city: res['cr_gdf'].to_crs(STORAGE_CRS) for city, res in all_zone_results.items()}

_title = (f"Study Area: {_n_shown} Nigerian Cities ({config.CITY_SELECTION_MODE})"
          + (f" — {_n_absorbed} absorbed, populations rolled up" if _n_absorbed else ""))

fig_map = create_nigeria_cities_landscape_map(
    aoi_dict=aoi_dict,
    cities_csv_path=config.get_aoi_data_path('cities_csv'),
    landscape_path=config.get_aoi_data_path('landscape_geojson'),
    nigeria_path=config.get_aoi_data_path('boundary_geojson'),
    cluster25_colors=config.CLUSTER25_COLORS,
    study_year=config.PROJECT_STUDY_YEAR,
    save_path=RESULTS_AOI / 'Nigeria_study_area_overview',
    title=_title,
    figsize=(12, 12),
    dpi=config.FIGURE_DPI,
    pop_scale=0.9,
    show_plot=True,
    pop_override=_pop_rollup,
)
plt.close(fig_map)
print(f"Saved: Nigeria_study_area_overview (.svg/.png/.pdf)")

# ── 8c: Standalone legend — proportional circle scale ────────────────────────
_pvals   = sorted(_pop_rollup.values())
_pop_min = _pvals[0]              if _pvals else 0
_pop_med = _pvals[len(_pvals)//2] if _pvals else 0
_pop_max = _pvals[-1]             if _pvals else 1

def _fmt(p):
    if p >= 1_000_000: return f'{p/1_000_000:.1f} M'
    if p >= 1_000:     return f'{p/1_000:.0f} k'
    return f'{p:.0f}'

POP_SCALE = 0.9
N_COLS    = 4
ROW_H     = 0.38
SEC_PAD   = 0.50

_cluster_items = sorted(config.CLUSTER25_COLORS.items())
n_rows_cl = int(np.ceil(len(_cluster_items) / N_COLS))
cl_h  = n_rows_cl * ROW_H + 0.55
lay_h = 2 * ROW_H + 0.40
circ_h = 1.65
fig_h = cl_h + SEC_PAD + lay_h + SEC_PAD + circ_h + 0.25
FIG_W = 9.0

fig_leg, ax = plt.subplots(figsize=(FIG_W, fig_h))
ax.set_xlim(0, FIG_W); ax.set_ylim(0, fig_h); ax.axis('off')
fig_leg.patch.set_facecolor('white')

def _hdr(y, txt):
    ax.text(0.15, y, txt, fontsize=10, fontweight='bold', va='top', ha='left', color='#222')
    ax.axhline(y - 0.21, xmin=0.02, xmax=0.98, color='#bbb', linewidth=0.8)

# Section A — Cluster25
y0 = fig_h - 0.12
_hdr(y0, 'A.  Physiographic Clusters (Cluster25)')
y_cl = y0 - 0.37
for i, (cid, chex) in enumerate(_cluster_items):
    ri, ci = divmod(i, N_COLS)
    x = 0.18 + ci * (FIG_W / N_COLS)
    y = y_cl - ri * ROW_H
    ax.add_patch(mpatches.FancyBboxPatch(
        (x, y - 0.22), 0.36, 0.26, boxstyle='round,pad=0.02',
        facecolor=chex, edgecolor='#555', linewidth=0.6,
        transform=ax.transData, clip_on=False))
    ax.text(x + 0.45, y - 0.09, f'Cluster {cid:02d}',
            fontsize=8, va='center', ha='left', color='#333')

# Section B — Map layers
y_b = y_cl - n_rows_cl * ROW_H - SEC_PAD
_hdr(y_b, 'B.  Map Layers')
_layers = [('line', '#333333', 1.8, 'National boundary'),
           ('point', 'black',  5,   'City location')]
y_li = y_b - 0.40
for ltype, lcol, lsz, llbl in _layers:
    if ltype == 'line':
        ax.plot([0.18, 0.68], [y_li, y_li], color=lcol, linewidth=lsz, solid_capstyle='round')
    else:
        ax.plot(0.43, y_li, 'o', color=lcol, markersize=lsz, zorder=5)
    ax.text(0.82, y_li, llbl, fontsize=8.5, va='center', ha='left', color='#222')
    y_li -= ROW_H

# Section C — Proportional circle scale
y_c = y_li - SEC_PAD + 0.05
_hdr(y_c, f'C.  Proportional Circle Scale — Population ({_n_shown} cities, absorbed rolled up)')
_entries = [(_pop_min, f'Min\n{_fmt(_pop_min)}'),
            (_pop_med, f'Median\n{_fmt(_pop_med)}'),
            (_pop_max, f'Max\n{_fmt(_pop_max)}')]
_xpos = [1.3, 4.0, 7.2]
R_MAX = 0.62
y_cc  = y_c - 1.08
for (pv, plbl), xp in zip(_entries, _xpos):
    r = (POP_SCALE * np.sqrt(pv / max(_pop_max, 1))) * R_MAX / POP_SCALE if pv > 0 else 0.04
    ax.add_patch(plt.Circle((xp, y_cc), r,
                             facecolor='#555555', edgecolor='white',
                             linewidth=0.8, alpha=0.75, zorder=4))
    ax.plot(xp, y_cc, 'ko', markersize=3, zorder=6)
    ax.annotate('', xy=(xp+r, y_cc), xytext=(xp-r, y_cc),
                arrowprops=dict(arrowstyle='<->', color='#555', lw=0.8,
                                shrinkA=0, shrinkB=0), zorder=5)
    ax.text(xp, y_cc - r - 0.14, plbl,
            ha='center', va='top', fontsize=8, color='#222', linespacing=1.2)

fig_leg.suptitle(f'Legend — Nigeria Study Area Overview ({config.PROJECT_STUDY_YEAR})',
                 fontsize=11, fontweight='bold', y=1.0)
fig_leg.tight_layout(rect=[0, 0, 1, 0.99])

_leg_base = RESULTS_AOI / 'Nigeria_study_area_legend'
fig_leg.savefig(_leg_base.with_suffix('.svg'),             bbox_inches='tight', facecolor='white')
fig_leg.savefig(_leg_base.with_suffix('.png'), dpi=config.FIGURE_DPI, bbox_inches='tight', facecolor='white')
fig_leg.savefig(_leg_base.with_suffix('.pdf'), format='pdf', bbox_inches='tight', facecolor='white')
plt.show()
print(f"Saved: Nigeria_study_area_legend (.svg/.png/.pdf)")

---
<a id="nb01-s09"></a>
## Section 9 — Population Trend Charts

**Purpose:** Visualise WUP 2024 population growth trajectories for all target cities.

**City set:** All cities in `cities_to_process` (determined by `config.CITY_SELECTION_MODE`). This includes both non-absorbed and absorbed cities — population trend data is city-level and is always valid regardless of AOI absorption status.

- **Composite chart** — all target cities overlaid on one figure; saved to `Results/01_aoi_delineation/`
- **Per-city charts** — one SVG per city; for cities with a zone result the chart is saved under their `{city}/figures/` folder; for absorbed cities a folder is created under `Results/01_aoi_delineation/{city_slug}/figures/`

---

> ↑ [Back to Table of Contents](#nb01-toc)

In [ ]:
from visualization import create_composite_population_trend
import pandas as pd

# Absorbed cities dict: {absorbed_city: parent_city}
_absorbed_df = pd.read_csv(RESULTS_AOI / 'absorbed_cities.csv')
_absorbed_map = dict(zip(_absorbed_df['absorbed_city'], _absorbed_df['parent_aoi']))

# Non-absorbed cities (those with their own zone result)
_non_absorbed = set(all_zone_results.keys())

# Build compat dict: one entry per non-absorbed city.
# Each parent bundles its own WUP row + any absorbed cities' rows so that
# mode='aoi' (sum over rows) gives the correct combined population per year.
_compat = {}
for _city in sorted(_non_absorbed):
    if _city not in cities_to_process:
        continue
    # Parent city's own row(s)
    _rows = cities_df[cities_df['Urban Agglomeration'] == _city].copy()
    # Append absorbed children's rows
    for _abs, _par in _absorbed_map.items():
        if _par == _city:
            _abs_rows = cities_df[cities_df['Urban Agglomeration'] == _abs]
            if not _abs_rows.empty:
                _rows = pd.concat([_rows, _abs_rows], ignore_index=True)
    if not _rows.empty:
        _compat[_city] = {'cities_in_aoi_df': _rows}

_n_absorbed = len(_absorbed_map)
print(f"City selection mode  : {config.CITY_SELECTION_MODE}")
print(f"AOI entries (lines)  : {len(_compat)}  (non-absorbed study cities)")
print(f"Absorbed folded in   : {_n_absorbed}  (population rolled up to parent)")
print()

fig_comp, stats_comp = create_composite_population_trend(
    _compat,
    save_dir=str(RESULTS_AOI),
    figsize=(12, 7),
    dpi=config.FIGURE_DPI,
    mode='aoi',
    target_cities=list(_compat.keys()),
)
plt.show()
plt.close(fig_comp)

yr0 = stats_comp.get('first_year', '?')
yr1 = stats_comp.get('last_year', '?')
print(f"Cities in composite  : {stats_comp['n_cities']}  (each parent includes absorbed)")
print(f"Population {yr0}    : {stats_comp['pop_first_k']:,.0f} k")
print(f"Population 2024      : {stats_comp['pop_2024_k']:,.0f} k ({stats_comp['pop_2024_k']/1000:.1f} M)")
print(f"Population {yr1}    : {stats_comp['pop_last_k']:,.0f} k")
print(f"Trend slope          : {stats_comp['slope_M_per_yr']:,.3f} M/yr   R2 = {stats_comp['r_squared']:.4f}")
print(f"Saved to             : {RESULTS_AOI}")

In [ ]:
from visualization import create_population_trends_plot
import pandas as pd

_absorbed_df  = pd.read_csv(RESULTS_AOI / 'absorbed_cities.csv')
_absorbed_map = dict(zip(_absorbed_df['absorbed_city'], _absorbed_df['parent_aoi']))
_absorbed_set = set(_absorbed_map.keys())

# Only generate charts for non-absorbed cities (absorbed appear in parent chart).
# Each parent's chart shows combined population (parent + absorbed children).
_non_absorbed = [c for c in cities_to_process if c not in _absorbed_set]

print(f"Generating per-city trend charts ({config.CITY_SELECTION_MODE}):")
print(f"  Non-absorbed cities : {len(_non_absorbed)}  (each gets own chart)")
print(f"  Absorbed cities     : {len(_absorbed_set)}  (folded into parent charts, no separate file)")
print()
skipped = created = errors = 0

for _city in _non_absorbed:
    _city_slug = _city.replace(' ', '_')

    if _city in all_zone_results:
        _city_dir = all_zone_results[_city]['city_dir']
    else:
        _city_dir = RESULTS_AOI / _city_slug
        _city_dir.mkdir(parents=True, exist_ok=True)

    _fig_dir  = _city_dir / 'figures'
    _fig_dir.mkdir(parents=True, exist_ok=True)
    _fig_path = _fig_dir / f"{_city_slug}_population_trend.svg"

    if _fig_path.exists():
        skipped += 1
        continue

    # Combined DataFrame: parent row(s) + absorbed children's rows
    _rows = cities_df[cities_df['Urban Agglomeration'] == _city].copy()
    for _abs, _par in _absorbed_map.items():
        if _par == _city:
            _abs_rows = cities_df[cities_df['Urban Agglomeration'] == _abs]
            if not _abs_rows.empty:
                _rows = pd.concat([_rows, _abs_rows], ignore_index=True)

    if _rows.empty:
        errors += 1
        print(f"  WARNING {_city}: not found in cities_df")
        continue

    _aoi_results = {'cities_in_aoi_df': _rows}
    try:
        fig_pop = create_population_trends_plot(
            _aoi_results, _city,
            save_path=str(_fig_path),
            figsize=(8, 5),
        )
        if fig_pop:
            plt.close(fig_pop)
            created += 1
    except Exception as e:
        print(f"  WARNING {_city}: {e}")
        errors += 1

print(f"Done — {created} created, {skipped} cached, {errors} errors.")

---
<a id="nb01-s10"></a>
## Section 10 — GEE Export Trigger

Exports each city's three zones as a **per-city GEE FeatureCollection asset**.
One task per city; each asset contains 3 features filtered by `zone` property.

```
Asset structure
───────────────
projects/{project_id}/assets/{collection_base}/{city_slug}
  Feature 1 → zone = 'CityRegion'   (full raster extraction domain)
  Feature 2 → zone = 'UrbanCore'    (UC, water-free)
  Feature 3 → zone = 'BRA'          (Background Reference Area, water-free)
```

**Loading a city in downstream notebooks (NB03 – NB06):**
```python
fc = ee.FeatureCollection("projects/{project_id}/assets/{collection_base}/{city_slug}")
cr = fc.filter(ee.Filter.eq("zone", "CityRegion")).first().geometry()
uc = fc.filter(ee.Filter.eq("zone", "UrbanCore")).first().geometry()
```

**Run once after AOI changes; skip if assets are already up to date.**


---

> ↑ [Back to Table of Contents](#nb01-toc)

In [ ]:
# ✅ SPEC CONFIRMED (§2.1): Per-city GEE export — one FeatureCollection asset per city,
# containing 3 features (UC, BRA, CityRegion). Future notebooks load by city slug.
#
# Asset path: projects/{project_id}/assets/{collection_base}/{city_slug}
# Each feature has a 'zone' property ('UrbanCore' | 'BRA' | 'CityRegion') for filtering.
#
# Geometry fixes applied before serialisation:
#   .buffer(0)       — repair topology (GEOSException fix)
#   .simplify(0.001) — reduce vertex count (~100 m tolerance, within SMOD 1 km resolution)

if not GEE_AVAILABLE:
    print("⚠️ GEE not available — skipping export")
else:
    collection_base = config.GEE_CONFIG.get('collection_name', 'Nigeria_AOI')
    project_id      = config.GEE_CONFIG['project_id']
    export_date     = datetime.utcnow().isoformat()

    submitted = skipped = errors = 0

    for city, res in all_zone_results.items():
        city_slug = res['city_slug']
        meta      = res['metadata']
        asset_id  = f"projects/{project_id}/assets/{collection_base}/{city_slug}"

        # Build 3 features for this city (one per zone)
        features = []
        for zone_key, zone_label in [('cr_gdf', 'CityRegion'),
                                      ('uc_gdf', 'UrbanCore'),
                                      ('bra_gdf', 'BRA')]:
            try:
                gdf = res[zone_key].to_crs(STORAGE_CRS).copy()
                gdf['geometry'] = gdf.geometry.buffer(0).simplify(0.001)
                geom = gdf.geometry.unary_union
                feat = ee.Feature(
                    ee.Geometry(mapping(geom)),
                    {
                        'city':           city,
                        'city_slug':      city_slug,
                        'zone':           zone_label,
                        'uc_area_km2':    meta.get('uc_area_km2', 0),
                        'bra_area_km2':   meta.get('bra_area_km2', 0),
                        'cr_area_km2':    meta.get('cr_area_km2', 0),
                        'bra_buffer_m':   meta.get('bra_buffer_m', config.BRA_BUFFER_M),
                        'bra_method':     meta.get('bra_method', 'fixed_buffer'),
                        'source_version': meta.get('source_version', ''),
                        'exported_date':  export_date,
                    }
                )
                features.append(feat)
            except Exception as e:
                print(f"  WARNING [{city}] {zone_label}: {e}")

        if len(features) < 3:
            print(f"  SKIP [{city}]: only {len(features)}/3 zones built")
            errors += 1
            continue

        try:
            fc   = ee.FeatureCollection(features)
            task = ee.batch.Export.table.toAsset(
                collection=fc,
                description=f"NB01_{city_slug}",
                assetId=asset_id,
            )
            task.start()
            submitted += 1
        except Exception as e:
            print(f"  ERROR [{city}]: {e}")
            errors += 1

    print(f"Export tasks submitted : {submitted}")
    print(f"Errors / skipped       : {errors}")
    print(f"Monitor at: https://code.earthengine.google.com/tasks")
    print(f"To load a city in future notebooks:")
    print(f'  fc = ee.FeatureCollection("projects/{project_id}/assets/{collection_base}/{{city_slug}}")')
    print(f'  cr = fc.filter(ee.Filter.eq("zone", "CityRegion")).first().geometry()')


---
<a id="nb01-s11"></a>
## Section 11 — Predominant Cluster25 per Study City

**Purpose:** Determine the **predominant physiographic landscape class (Cluster25)** for each study city by spatially intersecting the city's City-Region boundary with the national landscape character polygon dataset. This cluster attribution is the primary grouping variable in NB03–NB06.

> The Nigeria Cluster25 overview map is already in **Section 8**.
> This section adds a *city-level attribution* — which cluster covers the largest share
> of each city's study area — and a focused dot map showing that attribution.

**Why area-weighted modal assignment (not simple polygon count)?**
The Cluster25 polygons vary considerably in geographic extent. Counting polygons by frequency would bias the assignment toward whichever cluster happens to be subdivided into many small polygons, regardless of how much of the city's actual landscape it covers. Area-weighting ensures the assigned cluster genuinely dominates the land surface of the study area, which is the ecologically relevant criterion for linking LST patterns to landscape type.

**Inputs:**
- City-Region boundaries (`all_zone_results`, built in §4)
- National Cluster25 polygon dataset (loaded from `src/data_io.py`)

**Method:** For each city AOI, intersect Cluster25 polygons (reprojected to ESRI:102022) with the City-Region boundary and sum the intersection area per Cluster25 class. The class with the **largest total intersected area** is the predominant cluster for that city.

**Outputs:**
- `Results/01_aoi_delineation/city_cluster25_lookup.csv` — per-city lookup (Cluster25, area, climate/geology/elevation class)
- `Results/01_aoi_delineation/cluster25_city_summary.csv` — cluster-level frequency summary
- `Results/01_aoi_delineation/cities_cluster25_dot_map.svg/.pdf` — city dots coloured by predominant cluster

---

> ↑ [Back to Table of Contents](#nb01-toc)

In [ ]:
import geopandas as gpd
import pandas as pd
import shapely as _shp
import re as _re
from pathlib import Path

# ── Load landscape polygons (city CR AOIs are in memory via all_zone_results) ──
_lp = config.LANDSCAPE_DATA_PATH
_bp = config.get_aoi_data_path('boundary_geojson')
_landscape = gpd.read_file(_lp)              # EPSG:3857
_nigeria    = gpd.read_file(_bp).to_crs('EPSG:3857')

# ── Build city CR AOIs — validate + snap before union ─────────────────────────
# shapely 2.x: make_valid repairs self-intersecting rings; set_precision snaps
# coordinates to a 1 cm grid, eliminating floating-point topology conflicts.
_GRID = 0.01   # 1 cm in EPSG:3857

_aoi_rows = []
for _city, _res in all_zone_results.items():
    _cr = _res['cr_gdf'].to_crs('EPSG:3857')
    _clean = _shp.set_precision(_shp.make_valid(_cr.geometry.values), grid_size=_GRID)
    _aoi_rows.append({'city_name': _city, 'geometry': _shp.union_all(_clean)})
_cities_aoi = gpd.GeoDataFrame(_aoi_rows, crs='EPSG:3857')
print(f'City AOIs: {len(_cities_aoi)}')

# ── Apply same fixes to landscape polygons ────────────────────────────────────
_landscape_ov = _landscape.copy()
_landscape_ov['geometry'] = _shp.set_precision(
    _shp.make_valid(_landscape_ov.geometry.values), grid_size=_GRID)

# ── Spatial intersection — manual per-pair loop ───────────────────────────────
# gpd.overlay can raise GEOSException even after make_valid/set_precision on some
# geometry pairs; per-pair try/except lets us skip problem cases gracefully.
_int_rows = []
_skip_count = 0
for _, _cr in _cities_aoi.iterrows():
    _cname, _cgeom = _cr['city_name'], _cr['geometry']
    _candidates = _landscape_ov[_landscape_ov.intersects(_cgeom)]
    for _, _lr in _candidates.iterrows():
        try:
            _inter = _cgeom.intersection(_lr['geometry'])
        except Exception:
            try:
                _inter = _cgeom.buffer(0).intersection(_lr['geometry'].buffer(0))
            except Exception:
                _skip_count += 1
                continue
        if _inter.is_empty:
            continue
        _area = _inter.area / 1e6
        if _area < 1e-9:
            continue
        _int_rows.append({'city_name': _cname, 'Cluster25': _lr['Cluster25'],
                          'intersect_area_km2': _area})
if _skip_count:
    print(f'Warning: {_skip_count} polygon pairs skipped (unresolvable topology)')
_jn = pd.DataFrame(_int_rows)

# ── Predominant cluster per city (largest intersection area) ──────────────────
_dom = (_jn.groupby(['city_name','Cluster25'])['intersect_area_km2']
        .sum().reset_index()
        .sort_values(['city_name','intersect_area_km2'], ascending=[True, False]))
city_cluster_lookup = _dom.drop_duplicates(subset='city_name', keep='first').copy()

_cmeta = (_landscape.groupby('Cluster25')
          .agg(land_code25    =('land_code25',    'first'),
               Climate_Class  =('Climate_Class',  'first'),
               Landcover_Class=('Landcover_Class','first'),
               Elevation_Class=('Elevation_Class','first'),
               Geology_Class  =('Geology_Class',  'first'),
               description    =('description',    'first'))
          .reset_index())
city_cluster_lookup = city_cluster_lookup.merge(_cmeta, on='Cluster25', how='left')
city_cluster_lookup.to_csv(
    Path(config.PROJECT_ROOT) / 'Results/01_aoi_delineation/city_cluster25_lookup.csv',
    index=False)
print(f'Lookup saved: city_cluster25_lookup.csv  ({len(city_cluster_lookup)} cities)')

# ── Helper: format elevation label to include range ───────────────────────────
# Raw data: 'Very Low (0-100m)'  →  output: 'Very Low elevation (0-100m)'
def _elev_fmt(elev_class):
    s = str(elev_class).strip()
    m = _re.match(r'^(.*?)\s*(\(.*\))$', s)
    if m:
        return f"{m.group(1).strip()} elevation {m.group(2)}"
    return f"{s} elevation"

# ── TABLE 1 ───────────────────────────────────────────────────────────────────
# Columns: S/No. | City Regions | Physiographic Cluster | Frequency | Percentage (%) | Description
_n_total = len(city_cluster_lookup)
_rows = []
for _cid, _grp in city_cluster_lookup.groupby('Cluster25'):
    _cities = sorted(_grp['city_name'].tolist())
    _m = _grp.iloc[0]
    _elev_str = _elev_fmt(_m['Elevation_Class'])
    _desc = (f"{_m['Climate_Class']}, {_m['Geology_Class']}, "
             f"and {_elev_str.lower()}")
    _rows.append({
        'City Regions':          ', '.join(_cities),
        'Physiographic Cluster': f"C{int(_cid)}",
        'Frequency':             len(_cities),
        'Percentage (%)':        round(len(_cities) / _n_total * 100, 1),
        'Description':           _desc,
    })
cluster25_city_summary = (pd.DataFrame(_rows)
                          .sort_values('Frequency', ascending=False)
                          .reset_index(drop=True))
cluster25_city_summary.insert(0, 'S/No.', range(1, len(cluster25_city_summary) + 1))

cluster25_city_summary.to_csv(
    Path(config.PROJECT_ROOT) / 'Results/01_aoi_delineation/cluster25_city_summary.csv',
    index=False)
print(f'Summary CSV saved: cluster25_city_summary.csv  ({len(cluster25_city_summary)} clusters)')

print(f'\nTABLE 1 \u2014 Predominant Physiographic Cluster '
      f'({_n_total} cities, {len(cluster25_city_summary)} clusters)')
try:
    from IPython.display import display
    _s = (cluster25_city_summary.style
          .set_caption('Table 1: Predominant Physiographic Cluster (Cluster25) per City Region')
          .set_table_styles([
              {'selector': 'th', 'props': [('background-color','#404040'),
                                            ('color','white'),('font-weight','bold')]},
              {'selector': 'caption',
               'props': [('font-weight','bold'),('font-size','13px'),('text-align','left')]},
          ])
          .set_properties(**{'font-size': '11px'})
          .set_properties(subset=['City Regions','Description'],
                          **{'white-space':'normal','text-align':'left','max-width':'360px'})
          .set_properties(subset=['Physiographic Cluster','Frequency','Percentage (%)'],
                          **{'text-align':'center'})
          .bar(subset=['Percentage (%)'], color='#4472C4', vmin=0, vmax=100))
    display(_s)
except Exception:
    print(cluster25_city_summary.to_string(index=False))


---
<a id="nb01-s11a"></a>
### §11a — City Population from WUP (2024 Reference)

**Purpose:** Build a per-city WUP 2024 population lookup for all non-absorbed study cities. This is used as the population reference in cluster aggregates (§11b) and the study-city list (§11c).

**Why WUP, not GHSL?**
Splitting population into UC / BRA / CR zones requires a spatially-extracted population raster (GHSL). That extraction is performed in **NB05**, where it is needed for hotspot weighting. For NB01, which focuses on AOI delineation and cluster assignment, the WUP city-level totals are sufficient and reproducible without a live GEE session.

**Input:** `cities_df` — UN WUP 2024 population table (loaded in §2)

**Output:** `city_pop_wup` — dict `{city_name: population_2024_wup_k}` (population in thousands, 2024)

In [ ]:
# §11a — City population from WUP (no GEE required)
import pandas as pd

_yr = str(config.PROJECT_STUDY_YEAR)

# Build lookup for all non-absorbed cities (keys = all_zone_results)
city_pop_wup = {}
for _city in all_zone_results:
    _row = cities_df[cities_df['Urban Agglomeration'] == _city]
    if not _row.empty and _yr in cities_df.columns and pd.notna(_row[_yr].values[0]):
        city_pop_wup[_city] = float(_row[_yr].values[0])   # already in thousands
    else:
        city_pop_wup[_city] = None

_found = sum(1 for v in city_pop_wup.values() if v is not None)
print(f"WUP {_yr} population loaded for {_found}/{len(city_pop_wup)} non-absorbed cities.")
print()
print(f"  {'City':<24}  {'WUP 2024 (k)':>13}")
print("  " + "─" * 40)
for _c, _p in list(city_pop_wup.items())[:10]:
    _ps = f"{_p:>13,.1f}k" if _p is not None else f"{'N/A':>13}"
    print(f"  {_c:<24}  {_ps}")
if len(city_pop_wup) > 10:
    print(f"  ... ({len(city_pop_wup) - 10} more cities)")

---
<a id="nb01-s11b"></a>
### §11b — Cluster Zone Area & Population Summary Table

**Purpose:** Build the primary cluster-level reference table combining zone areas based on WUP population estimates. This table is used across NB03–NB06 as the landscape reference.

**Inputs:**
- `zone_area_summary.csv` — per-city UC / BRA / CR areas
- `city_cluster25_lookup.csv` — per-city Cluster25 assignment
- `WUP_pop` dict — UC and CR population per city (from §11a)
- `cities_df` — 2024 WUP population estimates

**Method:** Inner join on `city_name`; aggregate to cluster level (sum areas and populations; list member cities). Only non-absorbed cities are included in cluster aggregates.

**Outputs:**
- `Results/01_aoi_delineation/zone_area_summary_with_pop.csv` — extended per-city table
- `Results/01_aoi_delineation/cluster_zone_area_summary.csv` — cluster-level aggregate table


In [ ]:
# §11b — Cluster Zone Area & Population Summary Table (WUP 2024)
import pandas as pd
from pathlib import Path

RESULTS_AOI = Path(config.PROJECT_ROOT) / 'Results' / '01_aoi_delineation'

# ── 1. Load base tables ───────────────────────────────────────────────────────
_area_df     = pd.read_csv(RESULTS_AOI / 'zone_area_summary.csv')
_clust_df    = pd.read_csv(RESULTS_AOI / 'city_cluster25_lookup.csv')
_absorbed_df = pd.read_csv(RESULTS_AOI / 'absorbed_cities.csv')
_absorbed_map = dict(zip(_absorbed_df['absorbed_city'], _absorbed_df['parent_aoi']))

# ── 2. WUP 2024 population for all target cities ──────────────────────────────
_yr  = str(config.PROJECT_STUDY_YEAR)
_wup = (cities_df[['Urban Agglomeration', _yr]]
        .rename(columns={'Urban Agglomeration': 'city',
                         _yr: 'population_wup_k'})
        .copy())
_wup['population_wup_k'] = pd.to_numeric(_wup['population_wup_k'], errors='coerce')

# ── 3. Build extended per-city table (non-absorbed only) ──────────────────────
_ext = (_area_df
        .merge(_clust_df[['city_name', 'Cluster25', 'Climate_Class',
                           'Geology_Class', 'Elevation_Class']],
               left_on='city', right_on='city_name', how='left')
        .drop(columns='city_name')
        .merge(_wup, on='city', how='left'))

_ext['is_absorbed'] = _ext['absorbed_by'].notna() & (_ext['absorbed_by'] != '')
_active = _ext[~_ext['is_absorbed']].copy()

# ── 4. Roll up absorbed-city WUP population into parent rows ──────────────────
_rollup_notes = []
for _abs_city, _par_city in _absorbed_map.items():
    _abs_pop = _wup.loc[_wup['city'] == _abs_city, 'population_wup_k']
    if _abs_pop.empty:
        continue
    _abs_pop_k = float(_abs_pop.values[0])
    _mask = _active['city'] == _par_city
    if _mask.any():
        _old = float(_active.loc[_mask, 'population_wup_k'].values[0])
        _active.loc[_mask, 'population_wup_k'] = _old + _abs_pop_k
        _rollup_notes.append(f"  {_abs_city:22s} -> {_par_city}  (+{_abs_pop_k:,.1f}k)")

if _rollup_notes:
    print(f"WUP roll-up ({len(_rollup_notes)} absorbed cities):")
    for _n in _rollup_notes:
        print(_n)
    print()

_ext.to_csv(RESULTS_AOI / 'zone_area_summary_with_pop.csv', index=False)
print(f"Saved zone_area_summary_with_pop.csv  ({len(_ext)} non-absorbed cities)")

# ── 5. Aggregate to cluster level ─────────────────────────────────────────────
_clust_rows = []
for _cid, _g in _active.groupby('Cluster25', sort=True):
    _cities = sorted(_g['city'].tolist())
    _clust_rows.append({
        'Cluster25':             int(_cid),
        'n_cities':              len(_g),
        'cities':                ', '.join(_cities),
        'total_cr_area_km2':     round(_g['cr_area_km2'].sum(),  2),
        'total_bra_area_km2':    round(_g['bra_area_km2'].sum(), 2),
        'total_uc_area_km2':     round(_g['uc_area_km2'].sum(),  2),
        'total_population_wup_k': round(_g['population_wup_k'].sum(), 1)
                                  if _g['population_wup_k'].notna().any() else None,
        'climate_class':   _g['Climate_Class'].mode().iat[0]
                           if not _g['Climate_Class'].isna().all() else None,
        'geology_class':   _g['Geology_Class'].mode().iat[0]
                           if not _g['Geology_Class'].isna().all() else None,
        'elevation_class': _g['Elevation_Class'].mode().iat[0]
                           if not _g['Elevation_Class'].isna().all() else None,
    })

cluster_zone_summary = (pd.DataFrame(_clust_rows)
                         .sort_values('n_cities', ascending=False)
                         .reset_index(drop=True))
cluster_zone_summary.to_csv(RESULTS_AOI / 'cluster_zone_area_summary.csv', index=False)
print(f"Saved cluster_zone_area_summary.csv  ({len(cluster_zone_summary)} clusters)")
print()

# ── 6. Display table ──────────────────────────────────────────────────────────
print("Cluster Zone & Population Summary (WUP 2024, absorbed rolled up):")
print(f"{'Clust':>6}  {'N':>3}  {'UC km2':>9}  {'BRA km2':>9}  "
      f"{'CR km2':>9}  {'WUP 2024 (k)':>13}")
print("─" * 58)
for _, _r in cluster_zone_summary.iterrows():
    _wp = f"{_r['total_population_wup_k']:>13,.1f}" if pd.notna(_r['total_population_wup_k']) else f"{'N/A':>13}"
    print(f"  C{int(_r['Cluster25']):>2}   {int(_r['n_cities']):>3}  "
          f"{_r['total_uc_area_km2']:>9,.1f}  {_r['total_bra_area_km2']:>9,.1f}  "
          f"{_r['total_cr_area_km2']:>9,.1f}  {_wp}")
print()
_tot = cluster_zone_summary['total_population_wup_k'].sum()
print(f"National total (WUP 2024): {_tot:,.1f}k  =  {_tot/1_000:.2f} M")
print(f"Clusters: {len(cluster_zone_summary)}  |  AOI cities: {cluster_zone_summary['n_cities'].sum()}")

---
<a id="nb01-s11c"></a>
### §11c — Definitive Study City List

**Purpose:** Save a single authoritative CSV listing every city considered by this study together with its status. This file is the canonical reference for all downstream notebooks when checking which cities to include.

**Status values:**
| Status | Meaning |
|--------|---------|
| `included` | Active analysis city — has an independent AOI and meets the 300k threshold |
| `absorbed` | City whose AOI falls within another city's AOI — excluded from spatial analyses |

> **Note for NB02:** A further status value `excluded_stakeholder_below_300k` will be appended to this file in NB02 §3 when stakeholder records from below-300k cities are identified and removed from the perception analysis.


In [ ]:
# §11c — Definitive Study City List (study_cities_final.csv)
import pandas as pd
from pathlib import Path

RESULTS_AOI = Path(config.PROJECT_ROOT) / 'Results' / '01_aoi_delineation'

# ── Non-absorbed cities ────────────────────────────────────────────────────────
_base = pd.read_csv(RESULTS_AOI / 'zone_area_summary_with_pop.csv')
_base['status'] = 'included'

_cols_keep = ['city', 'population_wup_k', 'Cluster25',
              'uc_area_km2', 'bra_area_km2', 'cr_area_km2',
              'absorbed_by', 'status']
_final = (_base[[c for c in _cols_keep if c in _base.columns]]
          .rename(columns={'city': 'city_name', 'Cluster25': 'cluster25',
                           'population_wup_k': 'population_2024_wup_k'}))

# ── Add absorbed cities as explicit rows ──────────────────────────────────────
_absorbed_df  = pd.read_csv(RESULTS_AOI / 'absorbed_cities.csv')
_absorbed_map = dict(zip(_absorbed_df['absorbed_city'], _absorbed_df['parent_aoi']))
_parent_clust = _final.set_index('city_name')['cluster25'].to_dict()
_yr           = str(config.PROJECT_STUDY_YEAR)

_abs_rows = []
for _abs_city, _par_city in _absorbed_map.items():
    _wup_row = cities_df[cities_df['Urban Agglomeration'] == _abs_city]
    _pop_k   = float(_wup_row[_yr].values[0]) if not _wup_row.empty else None
    _abs_rows.append({
        'city_name':              _abs_city,
        'population_2024_wup_k':  _pop_k,
        'cluster25':              _parent_clust.get(_par_city),
        'uc_area_km2':            None,
        'bra_area_km2':           None,
        'cr_area_km2':            None,
        'absorbed_by':            _par_city,
        'status':                 'absorbed',
    })

_final = (pd.concat([_final, pd.DataFrame(_abs_rows)], ignore_index=True)
          .sort_values(['status', 'city_name'])
          .reset_index(drop=True))

_final.to_csv(RESULTS_AOI / 'study_cities_final.csv', index=False)

_inc = (_final['status'] == 'included').sum()
_abs = (_final['status'] == 'absorbed').sum()
print(f"study_cities_final.csv  —  {_inc} included  +  {_abs} absorbed  =  {len(_final)} total")
print()
print("Absorbed cities (inherit parent cluster):")
for _, _r in _final[_final['status'] == 'absorbed'].iterrows():
    _cid = f"C{int(_r['cluster25'])}" if pd.notna(_r['cluster25']) else 'N/A'
    _pk  = f"{_r['population_2024_wup_k']:,.0f}k" if pd.notna(_r['population_2024_wup_k']) else 'N/A'
    print(f"  {_r['city_name']:<22s} -> {_r['absorbed_by']:<22s}  cluster {_cid}  {_pk}")
print()
_show = _final[_final['status'] == 'included'][['city_name','cluster25','population_2024_wup_k']]
print(f"Included cities ({_inc}):")
print(f"  {'City':<24}  {'Cluster':>8}  {'WUP 2024'}")
print("  " + "─" * 44)
for _, _r in _show.iterrows():
    _cid = f"C{int(_r['cluster25'])}" if pd.notna(_r['cluster25']) else 'N/A'
    _pk  = f"{_r['population_2024_wup_k']:,.0f}k" if pd.notna(_r['population_2024_wup_k']) else 'N/A'
    print(f"  {_r['city_name']:<24}  {_cid:>8}  {_pk}")

---
<a id="nb01-s11d"></a>
### §11d — Population Trend Charts by Cluster & National

**Purpose:** Generate one WUP population trend chart per Cluster25 (same format as §9 per-city charts) and one national composite, so population growth trajectories can be compared across physiographic landscape types.

**Logic:**
- Each cluster chart aggregates the WUP time-series of all cities assigned to that cluster, **including absorbed cities** (rolled up into their parent for the total line, but listed individually in the breakdown box).
- The national chart aggregates all target study cities.

**Inputs:** `cluster_zone_summary` (from §11b), `cities_df` (WUP data), `absorbed_cities.csv`

**Outputs:**
- `Results/01_aoi_delineation/cluster_trends/cluster_<N>_population_trend.svg`
- `Results/01_aoi_delineation/cluster_trends/national_population_trend.svg`

In [ ]:
from visualization import create_population_trends_plot
import pandas as pd
from pathlib import Path
from collections import defaultdict

# Output directory for cluster trends
_trend_dir = RESULTS_AOI / 'cluster_trends'
_trend_dir.mkdir(parents=True, exist_ok=True)

# Absorbed map: {absorbed_city: parent_city}
_abs_df  = pd.read_csv(RESULTS_AOI / 'absorbed_cities.csv')
_abs_map = dict(zip(_abs_df['absorbed_city'], _abs_df['parent_aoi']))
_abs_set = set(_abs_map.keys())

# Year columns in cities_df (e.g. '1950','1955',...,'2050')
_yr_cols = [c for c in cities_df.columns if c.isdigit()]

# Non-absorbed target cities only (56)
_non_abs = sorted(c for c in cities_to_process if c not in _abs_set)


def _build_agg_df(city_list):
    # One row per non-absorbed city; absorbed children summed into their parent row.
    rows = []
    for _city in city_list:
        _all = [_city] + [a for a, p in _abs_map.items() if p == _city]
        _src = cities_df[cities_df['Urban Agglomeration'].isin(_all)]
        if _src.empty:
            continue
        _agg = {c: _src[c].sum() for c in _yr_cols if c in _src.columns}
        _agg['Urban Agglomeration'] = _city
        rows.append(_agg)
    return pd.DataFrame(rows) if rows else pd.DataFrame()


# Cluster assignment: {city_name: cluster_id}
_lkp        = pd.read_csv(RESULTS_AOI / 'city_cluster25_lookup.csv')
_city_clust = dict(zip(_lkp['city_name'], _lkp['Cluster25']))

# Group non-absorbed cities by cluster (absorbed are rolled up via _build_agg_df)
_clust_cities = defaultdict(list)
for _city in _non_abs:
    _cid = _city_clust.get(_city)
    if _cid is not None:
        _clust_cities[int(_cid)].append(_city)

print(f"Generating cluster population trend charts ({len(_clust_cities)} clusters)...")
_skip = _created = _errors = 0

for _cid in sorted(_clust_cities.keys()):
    _members = _clust_cities[_cid]   # non-absorbed cities in this cluster
    _slug    = f"cluster_{_cid:02d}"
    _fp      = _trend_dir / f"{_slug}_population_trend.svg"

    if _fp.exists():
        _skip += 1
        continue

    _rows = _build_agg_df(_members)  # each parent has absorbed children rolled in
    if _rows.empty:
        _errors += 1
        continue

    _n_abs_in_cluster = sum(1 for a, p in _abs_map.items() if p in _members)
    _title = (f"Cluster {_cid:02d} — {len(_members)} cities"
              + (f" (+{_n_abs_in_cluster} absorbed)" if _n_abs_in_cluster else ""))

    _aoi = {'cities_in_aoi_df': _rows}
    try:
        _fig = create_population_trends_plot(
            _aoi,
            current_city=_title,
            save_path=str(_fp),
            figsize=(9, 5),
        )
        if _fig:
            import matplotlib.pyplot as plt
            plt.close(_fig)
            _created += 1
    except Exception as _e:
        print(f"  WARNING Cluster {_cid}: {_e}")
        _errors += 1

print(f"Cluster charts done — {_created} created, {_skip} cached, {_errors} errors.")
print()

# ── National composite ────────────────────────────────────────────────────────
_nat_fp = _trend_dir / 'national_population_trend.svg'
if _nat_fp.exists():
    print("National trend chart already exists — skipped.")
else:
    # 56 rows: each non-absorbed city with absorbed children rolled in
    _nat_df  = _build_agg_df(_non_abs)
    _nat_aoi = {'cities_in_aoi_df': _nat_df}
    _n_abs   = len(_abs_map)
    _title   = f"Nigeria — {len(_non_abs)} Study Cities (+{_n_abs} absorbed rolled up)"
    try:
        _fig_nat = create_population_trends_plot(
            _nat_aoi,
            current_city=_title,
            save_path=str(_nat_fp),
            figsize=(10, 6),
        )
        if _fig_nat:
            import matplotlib.pyplot as plt
            plt.close(_fig_nat)
            print(f"National trend chart saved: {_title}")
    except Exception as _e:
        print(f"WARNING national chart: {_e}")

print(f"Charts saved to: {_trend_dir}")

---

### Extended Tables: Detailed Distribution & Colour Reference

Two further tables characterise the Cluster25 landscape composition in detail across all 56 study cities:

| Table | Content |
|-------|--------|
| **Table 2** | Cluster25 % distribution within each city zone — one row per city per zone (City Region + Urban Core), one column per cluster class present |
| **Table 3** | Cluster25 reference palette — all classes present in the 56 study cities with canonical colours |

> **Reading Table 2:** Each city appears as two consecutive rows — one for its **City Region (CR)** and one for its **Urban Core (UC)**. Each column is one Cluster25 class present anywhere in the dataset (C1, C4, C6…). Values show the percentage of that zone’s total land area covered by that class. Blank cells mean the class is absent. The dominant class in a city’s CR row matches the entry in Table 1.

> ↑ [Back to Table of Contents](#nb01-toc)

In [ ]:
import pandas as pd
import geopandas as gpd
import shapely as _shp
import re as _re
from pathlib import Path
from IPython.display import display

_GRID = 0.01   # 1 cm in EPSG:3857

# ═══ TABLE 2 — Cluster25 distribution by city zone (CR + UC combined) ═════════════
# Columns: City Name | City Zone | C1 | C4 | C6 | ...
# Each city has two consecutive rows: City Region then Urban Core

def _build_pct_wide(jn_df, zone_label):
    """Build wide % pivot for one zone; returns df with City Name + City Zone."""
    total  = jn_df.groupby('city_name')['intersect_area_km2'].sum().rename('total')
    detail = jn_df.groupby(['city_name','Cluster25'])['intersect_area_km2'].sum().reset_index()
    detail = detail.join(total, on='city_name')
    detail['pct'] = (detail['intersect_area_km2'] / detail['total'] * 100).round(1)
    wide = (detail.pivot(index='city_name', columns='Cluster25', values='pct')
                  .fillna(0).reset_index()
                  .rename(columns={'city_name': 'City Name'}))
    wide.columns = (['City Name'] + [f'C{int(c)}' for c in wide.columns[1:]])
    wide.insert(1, 'City Zone', zone_label)
    return wide.sort_values('City Name').reset_index(drop=True)

def _style_combined(df, caption):
    """Style combined zone table — YlOrRd gradient; blank zeros."""
    id_cols  = ['City Name', 'City Zone']
    num_cols = [c for c in df.columns if c not in id_cols]
    fmt = {c: (lambda v: f'{v:.0f}%' if v > 0 else '') for c in num_cols}
    return (df.style
              .set_caption(caption)
              .set_table_styles([
                  {'selector': 'th',
                   'props': [('background-color','#404040'),('color','white'),
                             ('font-weight','bold'),('font-size','10px')]},
                  {'selector': 'caption',
                   'props': [('font-weight','bold'),('font-size','13px'),
                             ('text-align','left')]},
              ])
              .format(fmt)
              .background_gradient(subset=num_cols, cmap='YlOrRd', vmin=0, vmax=100)
              .set_properties(**{'font-size':'10px','text-align':'center'})
              .set_properties(subset=['City Name'],
                              **{'text-align':'left','white-space':'nowrap'})
              .set_properties(subset=['City Zone'],
                              **{'text-align':'center','font-style':'italic'}))

# ── Build CR wide (re-uses _jn from cell 27) ──────────────────────────────────
print('Building Table 2 — CR + UC Cluster25 distributions...')
_cr_wide = _build_pct_wide(_jn, 'City Region')
print(f'  CR rows: {len(_cr_wide)}')

# ── Compute UC intersections ───────────────────────────────────────────────────
_uc_rows = []
for _city, _res in all_zone_results.items():
    _uc_gdf = _res.get('uc_gdf')
    if _uc_gdf is None:
        continue
    _uc_3857 = _uc_gdf.to_crs('EPSG:3857')
    _clean   = _shp.set_precision(_shp.make_valid(_uc_3857.geometry.values), grid_size=_GRID)
    _geom    = _shp.union_all(_clean)
    if _geom is None or _geom.is_empty:
        continue
    _uc_rows.append({'city_name': _city, 'geometry': _geom})
_uc_aoi = gpd.GeoDataFrame(_uc_rows, crs='EPSG:3857')
print(f'  UC AOIs prepared: {len(_uc_aoi)}')

_uc_int_rows, _uc_skip = [], 0
for _, _ucr in _uc_aoi.iterrows():
    _cname, _ugeom = _ucr['city_name'], _ucr['geometry']
    for _, _lr in _landscape_ov[_landscape_ov.intersects(_ugeom)].iterrows():
        try:
            _inter = _ugeom.intersection(_lr['geometry'])
        except Exception:
            try:
                _inter = _ugeom.buffer(0).intersection(_lr['geometry'].buffer(0))
            except Exception:
                _uc_skip += 1
                continue
        if _inter is None or _inter.is_empty:
            continue
        _area = _inter.area / 1e6
        if _area < 1e-9:
            continue
        _uc_int_rows.append({'city_name': _cname, 'Cluster25': _lr['Cluster25'],
                             'intersect_area_km2': _area})
if _uc_skip:
    print(f'  Warning: {_uc_skip} UC polygon pairs skipped')
_jn_uc = pd.DataFrame(_uc_int_rows)
print(f'  UC intersection rows: {len(_jn_uc)}')

if not _jn_uc.empty:
    _uc_wide = _build_pct_wide(_jn_uc, 'Urban Core')
    # Align both tables to the same cluster column set
    _all_c = sorted(
        set(list(_cr_wide.columns[2:]) + list(_uc_wide.columns[2:])),
        key=lambda x: int(x[1:]))
    for _w in [_cr_wide, _uc_wide]:
        for _c in _all_c:
            if _c not in _w.columns:
                _w[_c] = 0.0
    _cr_wide = _cr_wide[['City Name', 'City Zone'] + _all_c]
    _uc_wide = _uc_wide[['City Name', 'City Zone'] + _all_c]
    # Interleave CR then UC for each city
    _zone_order = {'City Region': 0, 'Urban Core': 1}
    _combined = pd.concat([_cr_wide, _uc_wide]).copy()
    _combined['_z'] = _combined['City Zone'].map(_zone_order)
    _combined = (_combined.sort_values(['City Name', '_z'])
                          .drop('_z', axis=1)
                          .reset_index(drop=True))
    print(f'  Combined rows: {len(_combined)} ({len(_cr_wide)} cities × 2 zones)')
    try:
        display(_style_combined(_combined,
            'Table 2: Cluster25 Distribution by City Zone (% of zone land area)'))
    except Exception:
        print(_combined.to_string(index=False))
    _combined.to_csv(
        Path(config.PROJECT_ROOT) /
        'Results/01_aoi_delineation/cluster25_zone_distribution.csv',
        index=False)
    print(f'  Saved: cluster25_zone_distribution.csv  ({len(_combined)} rows)')
else:
    print('  No UC intersection data — displaying City Region only.')
    try:
        display(_style_combined(_cr_wide,
            'Table 2: Cluster25 Distribution by City Zone — City Region only'))
    except Exception:
        print(_cr_wide.to_string(index=False))
    _cr_wide.to_csv(
        Path(config.PROJECT_ROOT) /
        'Results/01_aoi_delineation/cluster25_zone_distribution.csv',
        index=False)
    print(f'  Saved: cluster25_zone_distribution.csv  ({len(_cr_wide)} rows)')

# ═══ TABLE 3 — Cluster25 Reference Palette ════════════════════════════════════════

C25_COLORS = config.CLUSTER25_COLORS
_study_clusters = sorted(int(c) for c in city_cluster_lookup['Cluster25'].dropna().unique())

def _elev_fmt(elev_class):
    """'Very Low (0-100m)' -> 'Very Low elevation (0-100m)'"""
    s = str(elev_class).strip()
    m = _re.match(r'^(.*?)\s*(\(.*\))$', s)
    if m:
        return f"{m.group(1).strip()} elevation {m.group(2)}"
    return f"{s} elevation"

_ref_rows = []
for _cid in _study_clusters:
    _m = _cmeta[_cmeta['Cluster25'] == _cid]
    if _m.empty:
        continue
    _m = _m.iloc[0]
    _elev_str = _elev_fmt(_m['Elevation_Class'])
    _ref_rows.append({
        'Cluster25':     f'C{_cid}',
        'Climate Class': _m['Climate_Class'],
        'Geology Class': _m['Geology_Class'],
        'Elevation':     _elev_str,
        'Landcover':     _m['Landcover_Class'],
        'Description':   (f"{_m['Climate_Class']}, {_m['Geology_Class']}, "
                          f"and {_elev_str.lower()}"),
        'Hex Colour':    C25_COLORS.get(_cid, '#CCCCCC'),
    })
cluster25_reference = pd.DataFrame(_ref_rows)

def _txt_clr(hx):
    h = hx.lstrip('#')
    r, g, b = int(h[:2], 16), int(h[2:4], 16), int(h[4:], 16)
    return '#000000' if (0.299*r + 0.587*g + 0.114*b) > 140 else '#ffffff'

def _colour_col(s):
    return [f'background-color:{v}; color:{_txt_clr(v)}; '
            f'text-align:center; font-weight:bold; border:1px solid #555' for v in s]

print('\nTABLE 3 — Cluster25 Reference Palette (16 classes across 56 study cities)')
try:
    display(cluster25_reference.style
            .set_caption('Table 3: Cluster25 Reference — Classes & Canonical Colours')
            .set_table_styles([
                {'selector': 'th',
                 'props': [('background-color','#404040'),('color','white'),
                           ('font-weight','bold')]},
                {'selector': 'caption',
                 'props': [('font-weight','bold'),('font-size','13px'),
                           ('text-align','left')]},
            ])
            .set_properties(**{'font-size':'11px'})
            .apply(_colour_col, subset=['Hex Colour'], axis=0))
except Exception:
    print(cluster25_reference.to_string(index=False))

cluster25_reference.to_csv(
    Path(config.PROJECT_ROOT) /
    'Results/01_aoi_delineation/cluster25_reference_palette.csv',
    index=False)
print(f'Saved: cluster25_reference_palette.csv  ({len(cluster25_reference)} clusters)')


---
<a id="nb01-s12"></a>
## Section 12 — Publication Figures (A4)

**Purpose:** Generate the five publication-ready figures for the NB01 methods paper.

| # | Figure | Format | Size |
|---|--------|--------|------|
| 1 | Nigeria overview — Cluster25 + proportional city pop circles | SVG + PDF | A4 portrait |
| 2 | TEST cities — 3-zone maps (Lagos, Abuja, Kano, Jos, Port Harcourt) | SVG + PDF | Half-A4 portrait |
| 3 | Composite population trend — all study cities | SVG + PDF | Half-A4 landscape |
| 4 | Cluster zone area + WUP population — dual-axis bar/line | SVG + PDF | Half-A4 landscape |
| 5 | All-cluster population trend lines | SVG + PDF | Half-A4 landscape |

**Output folder:** `Results/01_aoi_delineation/publication/`

---

> ↑ [Back to Table of Contents](#nb01-toc)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
from pathlib import Path
from visualization import (
    create_publication_nigeria_overview,
    create_publication_city_grid,
    create_composite_population_trend,
    create_cluster_zone_population_chart,
    create_cluster_multiline_trend,
)

RESULTS_PUB = RESULTS_AOI / 'publication'
RESULTS_PUB.mkdir(parents=True, exist_ok=True)

# Re-derive absorbed map (needed for Figs 4/5)
_abs_df_pub  = pd.read_csv(RESULTS_AOI / 'absorbed_cities.csv')
_abs_map_pub = dict(zip(_abs_df_pub['absorbed_city'], _abs_df_pub['parent_aoi']))
_abs_set_pub = set(_abs_map_pub.keys())

# Non-absorbed target cities (56)
_non_abs_pub = sorted(c for c in cities_to_process if c not in _abs_set_pub)

# Population roll-up for overview map (same as §8)
_yr_pub = str(config.PROJECT_STUDY_YEAR)
_pop_rollup_pub = {}
for _, _r in cities_df.iterrows():
    _cn = _r['Urban Agglomeration']
    if _cn in cities_to_process and pd.notna(_r.get(_yr_pub)):
        _pop_rollup_pub[_cn] = float(_r[_yr_pub]) * 1_000
for _abs, _par in _abs_map_pub.items():
    _abs_wup = cities_df.loc[cities_df['Urban Agglomeration'] == _abs, _yr_pub]
    if not _abs_wup.empty and pd.notna(_abs_wup.values[0]) and _par in _pop_rollup_pub:
        _pop_rollup_pub[_par] += float(_abs_wup.values[0]) * 1_000

# Active cluster IDs (from cluster_zone_summary produced in §11b)
_active_clusters_pub = sorted(int(x) for x in cluster_zone_summary['Cluster25'].unique())

print(f"Publication output: {RESULTS_PUB}")
print(f"Non-absorbed cities : {len(_non_abs_pub)}")
print(f"Active clusters     : {len(_active_clusters_pub)}")

---
<a id="nb01-s12a"></a>
### §12a — Figure 1: Nigeria Overview Map

**Purpose:** A4 portrait publication figure showing Cluster25 physiographic
classification as filled polygons with proportional city population circles
and a full cluster colour legend inside the map bounds.

**Inputs:** `all_zone_results` (centroids), WUP data, landscape GeoJSON, Nigeria boundary, `_pop_rollup_pub`

**Output:** `publication/pub_fig1_nigeria_overview.svg / .pdf`

In [ ]:
fig1 = create_publication_nigeria_overview(
    aoi_dict          = all_zone_results,
    cities_csv_path   = config.PROJECT_ROOT / config.AOI_DATA_PATHS["cities_csv"],
    landscape_path    = config.LANDSCAPE_DATA_PATH,
    nigeria_path      = config.GHSL_POP_CONFIG["boundary_geojson"],
    cluster25_colors  = config.CLUSTER25_COLORS,
    active_cluster_ids= _active_clusters_pub,
    study_year        = config.PROJECT_STUDY_YEAR,
    pop_override      = _pop_rollup_pub,
    save_path         = RESULTS_PUB / "pub_fig1_nigeria_overview",
    dpi               = config.FIGURE_DPI,
    figsize           = (8.27, 11.69),
)
plt.show()
plt.close(fig1)

---
<a id="nb01-s12b"></a>
### §12b — Figure 2: TEST Cities Three-Zone Map Grid

**Purpose:** Half-A4 portrait grid with one three-zone map per TEST city.
Each panel has its own scale bar and north arrow. One shared legend at
the bottom of the grid.

**Cities:** Lagos, Abuja, Kano, Jos, Port Harcourt (config.TEST_CITIES)

**Inputs:** `all_zone_results`, `landscape_gdf` (loaded in §7), cluster colours

**Output:** `publication/pub_fig2_city_grid.svg / .pdf`

In [ ]:
# landscape_gdf was loaded in §7 — ensure it's available
if 'landscape_gdf' not in dir() or landscape_gdf is None:
    import geopandas as gpd
    landscape_gdf = gpd.read_file(config.LANDSCAPE_DATA_PATH).to_crs('EPSG:4326')
    landscape_gdf["geometry"] = landscape_gdf.geometry.buffer(0)
    print("Re-loaded landscape_gdf for §12b")

# Display order: Kano | Abuja | Jos / Lagos | Port Harcourt | Legend
_DISPLAY_ORDER = ['Kano', 'Abuja', 'Jos', 'Lagos', 'Port Harcourt']
_test_with_zones = [c for c in _DISPLAY_ORDER if c in all_zone_results]
print(f"TEST cities in display order: {_test_with_zones}")

fig2 = create_publication_city_grid(
    city_names        = _test_with_zones,
    zone_results      = all_zone_results,
    results_aoi_path  = RESULTS_AOI,
    landscape_gdf     = landscape_gdf,
    cluster25_colors  = config.CLUSTER25_COLORS,
    storage_crs       = 'EPSG:4326',
    save_path         = RESULTS_PUB / 'pub_fig2_city_grid',
    dpi               = config.FIGURE_DPI,
    figsize           = (8.27, 5.845),
)
plt.show()
plt.close(fig2)

---
<a id="nb01-s12c"></a>
### §12c — Figure 3: Composite Population Trend

**Purpose:** Half-A4 landscape composite population trend for all 56 non-absorbed
study cities, with absorbed populations rolled up to parent cities.

**Inputs:** `cities_df`, `_non_abs_pub`, `_abs_map_pub`

**Output:** `publication/pub_fig3_composite_trend.svg / .pdf`

In [ ]:
from visualization import create_composite_population_trend
import shutil

# Build compat dict (parent + absorbed rolled up) — same as §9 composite
_compat_pub = {}
for _city in sorted(_non_abs_pub):
    _rows = cities_df[cities_df['Urban Agglomeration'] == _city].copy()
    for _a, _p in _abs_map_pub.items():
        if _p == _city:
            _ar = cities_df[cities_df['Urban Agglomeration'] == _a]
            if not _ar.empty:
                _rows = pd.concat([_rows, _ar], ignore_index=True)
    if not _rows.empty:
        _compat_pub[_city] = {"cities_in_aoi_df": _rows}

fig3, _stats3 = create_composite_population_trend(
    _compat_pub,
    save_dir      = str(RESULTS_PUB),
    figsize       = (11.69, 5.845),
    dpi           = config.FIGURE_DPI,
    mode          = "aoi",
    target_cities = list(_compat_pub.keys()),
)
# Rename to publication filename
for _ext in (".svg", ".pdf", ".png"):
    _src = RESULTS_PUB / f"composite_population_trend{_ext}"
    _dst = RESULTS_PUB / f"pub_fig3_composite_trend{_ext}"
    if _src.exists():
        shutil.move(str(_src), str(_dst))
        print(f"Renamed: {_src.name} -> {_dst.name}")

plt.show()
plt.close(fig3)
print(f"Cities: {_stats3.get('n_cities')}  |  Pop 2024: {_stats3.get('pop_2024_k',0)/1000:.2f} M")

---
<a id="nb01-s12d"></a>
### §12d — Figure 4: Cluster Zone Area & Population Chart

**Purpose:** Half-A4 landscape dual-axis grouped bar chart. Left y-axis shows
UC / BRA / CR zone areas (km²) per cluster as coloured grouped bars. Right
y-axis overlays WUP 2024 population as a line with diamond markers.

X-axis: `Cl-{id}({N})` labels sorted by cluster ID.

**Inputs:** `cluster_zone_summary` (from §11b)

**Output:** `publication/pub_fig4_cluster_zones.svg / .pdf`

In [ ]:
fig4 = create_cluster_zone_population_chart(
    cluster_zone_summary = cluster_zone_summary,
    save_path            = RESULTS_PUB / 'pub_fig4_cluster_zones',
    dpi                  = config.FIGURE_DPI,
    figsize              = (11.69, 5.845),
)
plt.show()
plt.close(fig4)

---
<a id="nb01-s12e"></a>
### §12e — Figure 5: All-Cluster Population Trend Lines

**Purpose:** Half-A4 landscape multi-line WUP population trend chart — one
line per active Cluster25. Absorbed cities are rolled up into their parent
before cluster aggregation.

**Design:**
- Unique colour per cluster (tab20 palette)
- 2024 and 2050: vertical dashed lines + diamond markers (no text labels)
- Legend (right): cluster label, 2024 pop, overall growth rate
- Breakdown box (bottom-right): top-5 clusters by 2024 pop

**Inputs:** `cluster_zone_summary`, `cities_df`, `_abs_map_pub`

**Output:** `publication/pub_fig5_cluster_trends.svg / .pdf`

In [ ]:
fig5 = create_cluster_multiline_trend(
    cluster_zone_summary = cluster_zone_summary,
    cities_df            = cities_df,
    absorbed_map         = _abs_map_pub,
    cluster25_colors     = config.CLUSTER25_COLORS,
    save_path            = RESULTS_PUB / 'pub_fig5_cluster_trends',
    dpi                  = config.FIGURE_DPI,
    figsize              = (11.69, 5.845),
)
plt.show()
plt.close(fig5)

---
<a id="nb01-s12"></a>
## Section 12 — Summary & Next Steps

### Outputs

```
Results/01_aoi_delineation/
├── study_cities_final.csv              ← §11c: definitive city list (status: included / absorbed)
├── absorbed_cities.csv                 ← §5:   cities sharing a parent AOI
├── all_cities_zone_metadata.json       ← §5:   per-city metadata dict
├── all_cities_zones_4326.geojson       ← §5:   all 3 zones for all cities (WGS84)
├── all_cities_zones_102022.geojson     ← §5:   same in analysis CRS
├── zone_area_summary.csv               ← §5:   UC / BRA / CR areas per city
├── zone_area_summary_with_pop.csv      ← §11b: extended per-city table + GHSL population
├── cluster_zone_area_summary.csv       ← §11b: cluster-level aggregate (areas + population)
├── city_ghsl_population.json           ← §11a: GHSL UC + CR population cache
├── city_cluster25_lookup.csv           ← §11:  predominant Cluster25 per city
├── cluster25_city_summary.csv          ← §11:  Table 1: cluster × city frequency
├── cluster25_zone_distribution.csv     ← §11:  Table 2: Cluster25 % by city zone (CR + UC)
├── cluster25_reference_palette.csv     ← §11:  Table 3: canonical hex colours per cluster
├── Nigeria_study_area_overview.svg     ← §8
├── Nigeria_study_area_legend.svg       ← §8
├── Composite_population_trend.svg      ← §9
├── cities_cluster25_dot_map.svg        ← §11
└── {city_slug}/
    ├── {city_slug}_urban_core_4326_v5.geojson
    ├── {city_slug}_urban_core_102022_v5.geojson
    ├── {city_slug}_bra_4326_v5.geojson
    ├── {city_slug}_bra_102022_v5.geojson
    ├── {city_slug}_city_region_4326_v5.geojson
    ├── {city_slug}_city_region_102022_v5.geojson
    ├── {city_slug}_zone_metadata_v5.json
    ├── {city_slug}_water_bodies_4326.geojson
    ├── osm/
    │   ├── {city_slug}_roads_4326.geojson
    │   └── {city_slug}_places_4326.geojson
    └── figures/
        ├── {city_slug}_three_zone_map.svg
        └── {city_slug}_population_trend.svg
```

### Cross-notebook Dependencies

| Notebook | Key NB01 files consumed |
|----------|------------------------|
| **NB02** Stakeholder Analysis | `study_cities_final.csv` (city filter reference) |
| **NB03** Temporal LST | `city_cluster25_lookup.csv`, per-city `*_city_region_4326_v5.geojson`, GEE assets |
| **NB04** Daily LST + Events | `zone_area_summary.csv`, per-city UC + BRA `_v5` GeoJSONs, GEE assets |
| **NB05** Hotspot Analysis | NB03 + NB04 outputs; `city_cluster25_lookup.csv` |
| **NB06** Statistical Modelling | NB05 outputs; `cluster_zone_area_summary.csv` |

---

> ↑ [Back to Table of Contents](#nb01-toc)

---

## Next → Notebook 02 — Stakeholder Analysis

Open **`Notebook/02_stakeholder_analysis.ipynb`**

Before opening NB02, confirm:
- [ ] `study_cities_final.csv` saved — NB02 uses this as the filter reference
- [ ] `city_cluster25_lookup.csv` saved — required by NB03 and NB06
- [ ] `cluster_zone_area_summary.csv` saved — used by NB06 for landscape reference
- [ ] GEE export tasks submitted via §10 (required before running NB03 §4)
